# Performing Construct Validation Tests of the new Textual MAP measures


This file serves to perform various construct validation tests, including convergent/concurrent, discriminant, and predictive validity. The content validity tests are already performed during the MAP measurement pipelines. See code files contained in the folders '02_01_01_Prompt_Engineering', '02_01_02_Fine_Tuning', '02_02_W2V_Approach' and '02_03_BoW_Approach', as well as the corresponding output file 'data/GLLM/validation_summary_total.xlsx' for more details.
 

<div class='alert-warning'>
Libraries
</div>
First, we Import all necessary libraries. These inlcude 'os' to set and handle working directories; 'pandas' and 'numpy' for data handling and calculations; 'pickle' to load and save the prepared data as memory efficient pickle files; 'lseg.data' to retrieve further information from the LSEG Datastram API; 'plotnine', 'matplotlib', and 'seaborn' to create plots/figures; 'sklearn', 'shap', and 'scipy' to handle the machine learning and feature importance analyses; as well as 'statsmodels' and 'pyfixest' packages to run regression analyses. 

In [ ]:
#!/usr/bin/env python3

# load standard packages for data handling
import os
import pandas as pd
import numpy as np
import pickle

# load joblib for parallel processing 
#(Not necessarily needed, but can be useful for speeding up random forest and permutation importance calculations)
from joblib import Parallel, delayed
import multiprocessing

# load lseg package for data retrieval
import lseg.data as rd

# load packages for data visualization
import plotnine
import matplotlib.pyplot as plt
import seaborn as sns

# load sklearn, shap and scipy packages to handle the machine learning and feature importance analyses
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import shap
from scipy.stats.mstats import winsorize
from scipy.stats import pearsonr

# load statsmodels and pyfixest packages to handle the regression analyses
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
import pyfixest as pf 

<div class='alert-warning'>
Set the working directory
</div>

In [ ]:
# Set working directory 
os.chdir('../../data')

# Creat a folder to save summary tables if it does not exist
tables_dir = 'Analyses_outputs/Tables'
if not os.path.exists(tables_dir):
    os.makedirs(tables_dir)

# Create directory for saving the figures if it does not exist
plots_dir = f'Analyses_outputs/Plots/Validation'
if not os.path.exists(plots_dir):
    os.makedirs(plots_dir)

## Retrieve additional data from LSEG and further data sources

<div class='alert-warning'>
Load the datasets with the newly developed MAP dimension measures
</div>

In [ ]:
# Load the corpus with the normalized MAP measures
# We can choose between the different measurement types by changing the 'measurement_approach', 'within_industry' and 'without_finance' variables below.

# Specifiy the measurement approach to use for the analysis. Options are: 'BoW', 'W2V_v1', 'W2V_v2', 'GLLM'
measurement_approach = 'W2V_v2'

# Specify whether to use MAP measures normalized across the whole sample (False) or within each industry (True)
within_industry = False  

# Specify whether to exclude firms operating in the Financing/Investment sector (true or false).
without_finance = False

suffix = (
    '_within_industry_normalized' if within_industry
    else '_without_finance_normalized' if without_finance
    else '_final'
)

measurement_type = f'Corpus_df_{measurement_approach}_dimension_scores{suffix}'
corpus_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/{measurement_type}.pkl')

# Remove resource intensive variables from the corpus dataframe to save memory
columns_to_remove = ['filing_text', 'MAP_token_count', 'results', 'results_FT', 'Cost_df', 'Cost_FT_df', 
                     'Investment_df', 'Investment_FT_df', 'Operations_df', 'Operations_FT_df', 
                     'Performance_df', 'Performance_FT_df', 'Risk_df', 'Risk_FT_df', 'Strategy_df', 'Strategy_FT_df', 
                     'Pricing_df', 'Pricing_FT_df', 'Budget_df', 'Budget_FT_df']

corpus_df = corpus_df.drop(columns=columns_to_remove, errors='ignore')

#The RIC codes are stored in the 'SP_500_CIK_final' pickle file. So load the file into a dataframe.
CIK_df = pd.read_pickle('SP_500_CIK_final.pkl')

<div class='alert-info'>
Step 1: Retrieve financial (Company Fundamentals) and ESG data. 
</div>

First, we need to retrieve the financial (Company Fundamentals) and ESG data from the LSEG API. (We just need to do this once. For the other datasets use the final pickle file below)

In [ ]:
# Retrieve the financial and ESG variables from the past 14 years from Refinitiv

# Load the variable codes from the csv file into a dataframe
LSEG_variable_list = pd.read_csv('External/LSEG_variables_list_final.csv', sep=';')

LSEG_variable_list = ['TR.HeadquartersCountry','TR.RegStateProvince', 'TR.IPODate', 'TR.CompanyIncorpDate', 'TR.F.PeriodEndDate', 
'TR.NAICSSector', 'TR.NAICSSectorAllCode', 'TR.NAICSSubsectorAllCode'] + LSEG_variable_list['LSEG_code'].tolist()

# Define the 'universe' (RIC codes) of companies to get data from
universe = CIK_df['Instrument'].unique().astype(str).tolist()

# Open a new Refinitiv session (make sure that the desktop App is open)
rd.open_session()

# Get the financial and ESG variables from Refinitiv for all companies in the universe for the past 14 years
variable_df = rd.get_data(
    universe = universe,
    fields = LSEG_variable_list,
    parameters = {
        'SDate': '0',
        'EDate': '-15',
        'Frq': 'FY',
        'CH': 'IN;Fd',
        'RH': 'date'
    }
)

rd.close_session()

# Save the retrieved data to a pickle file
variable_df.to_pickle('External/LSEG_variable_df_final.pkl')

# Now: Fill gaps in 'Date of Incorporation' by hand where possible


Second, after filling some gaps in the 'Date of Incorporation' variable, we will create some new measures.

In [ ]:
#Load the retrieved variable dataframe
variable_df = pd.read_pickle('External/LSEG_variable_df_final.pkl')

# transform date columns to date (year) format
variable_df['Year of Incorporation'] = pd.to_datetime(variable_df['Date of Incorporation'], errors='coerce').dt.year

variable_df['Period End Year'] = pd.to_datetime(variable_df['Period End Date'], errors='coerce').dt.year

variable_df['Period End Month'] = pd.to_datetime(variable_df['Period End Date'], errors='coerce').dt.month

variable_df['IPO Year'] = pd.to_datetime(variable_df['IPO Date'], errors='coerce').dt.year

# Create Age variable as difference between Period End Year and Year of Incorporation if Year of Incorporation smaller than IPO Year, otherwise use IPO Year

variable_df['Age'] = np.where(
    (variable_df['Year of Incorporation'].notna()) & 
    (variable_df['IPO Year'].notna()),
    np.where(
        variable_df['Year of Incorporation'] < variable_df['IPO Year'],
        variable_df['Period End Year'] - variable_df['Year of Incorporation'],
        variable_df['Period End Year'] - variable_df['IPO Year']
    ),
    np.where(
        variable_df['Year of Incorporation'].notna(),
        variable_df['Period End Year'] - variable_df['Year of Incorporation'],
        np.where(
            variable_df['IPO Year'].notna(),
            variable_df['Period End Year'] - variable_df['IPO Year'],
            np.nan
        )
    )
)

# Next we create a new variable that indicates the revenue growth of the company compared to the previous year. We calculate it as the percentage change in revenue from business activities - total compared to the previous year. We use the 'shift' function to get the revenue from the previous year for each company. If the revenue from the previous year is missing or zero, we set the revenue growth to missing in this case.

variable_df['Revenue_lagged'] =   variable_df.groupby('Instrument')['Revenue from Business Activities - Total'].shift(-1)  

variable_df['Revenue Growth'] = np.where(
    variable_df['Revenue_lagged'].notna() & (variable_df['Revenue_lagged'] != 0),
    (variable_df['Revenue from Business Activities - Total'] - variable_df['Revenue_lagged']) / variable_df['Revenue_lagged'] * 100,
    np.nan
)

# Next we get ROA_lag_t_minus_1, EPS_lag_t_minus_1, Gross_Margin_lag_t_minus_1, Operating_Margin_lag_t_minus_1
variable_df['ROA_lag_t_minus_1'] = variable_df.groupby('Instrument')['Return on Average Total Assets - %, TTM'].shift(-1)
variable_df['EPS_lag_t_minus_1'] = variable_df.groupby('Instrument')['EPS - Diluted - excl Exord Items Applicable to Common Total'].shift(-1)
variable_df['Gross_Margin_lag_t_minus_1'] = variable_df.groupby('Instrument')['Gross Profit Margin - %'].shift(-1)
variable_df['Operating_Margin_lag_t_minus_1'] = variable_df.groupby('Instrument')['Operating Margin - %'].shift(-1)

# Move the new columns to the front
cols = variable_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('Year of Incorporation')))
cols.insert(2, cols.pop(cols.index('Period End Year')))
cols.insert(3, cols.pop(cols.index('Period End Month')))
cols.insert(4, cols.pop(cols.index('IPO Year')))
cols.insert(5, cols.pop(cols.index('Age')))
cols.insert(6, cols.pop(cols.index('Revenue Growth')))
variable_df = variable_df[cols]

# Next we create a new variable that indicates the life cycle stage of the company based on the operating, financing, and investing cash flows. We define the following life cycle stages:
# 'Introduction' CF negative in operating and investing activities, and positive in financing activities;
# 'Growth' CF negative in investing activities, and positive in operating and financing activities;
# 'Mature' CF negative in investing and financing activities, and positive in operating activities;
# 'Decline' CF positive in investing activities, and negative in operating and financing activities;
# 'Revive' others.
# CF columns: 'Net Cash Flow from Operating Activities', 'Net Cash Flow from Financing Activities', 
# and 'Net Cash Flow from Investing Activities' 
# where CF are missing , we cannot determine the life cycle stage, so we will set it to missing in this case.

variable_df['Life Cycle Stage 1'] = np.where(
    variable_df['Net Cash Flow from Operating Activities'].isna() |
    variable_df['Net Cash Flow from Investing Activities'].isna() |
    variable_df['Net Cash Flow from Financing Activities'].isna(),
    None, 
    np.where(
        (variable_df['Net Cash Flow from Operating Activities'] < 0) &
        (variable_df['Net Cash Flow from Investing Activities'] < 0) &
        (variable_df['Net Cash Flow from Financing Activities'] > 0),
        'Birth',
        np.where(
            (variable_df['Net Cash Flow from Investing Activities'] < 0) &
            (variable_df['Net Cash Flow from Operating Activities'] > 0) &
            (variable_df['Net Cash Flow from Financing Activities'] > 0),
            'Growth',
            np.where(
                (variable_df['Net Cash Flow from Investing Activities'] < 0) &
                (variable_df['Net Cash Flow from Financing Activities'] < 0) &
                (variable_df['Net Cash Flow from Operating Activities'] > 0),
                'Mature',
                np.where(
                    (variable_df['Net Cash Flow from Investing Activities'] > 0) &
                    (variable_df['Net Cash Flow from Operating Activities'] < 0) &
                    (variable_df['Net Cash Flow from Financing Activities'] < 0),
                    'Decline',
                    'Revive'
                )
            )
        )
    )
)

# Next we create a new variable that indicates the life cycle stage of the company based on the age variable. We define the following life cycle stages:
# Introduction is Age<=5, Growth if 5<Age<=10, Mature if 10<Age<=15, Revive if 15<Age<=20, Decline if Age>20
# Note that if Age < 0 or Age is missing, we cannot determine the life cycle stage, so we will set it to  missing in this case.

variable_df['Life Cycle Stage 2'] = np.where(
    variable_df['Age'] < 0,
    None,
    np.where(
        variable_df['Age'] <= 5,
        'Birth',
        np.where(
            variable_df['Age'] <= 10,
            'Growth',
            np.where(
                variable_df['Age'] <= 15,
                'Mature',
                np.where(
                    variable_df['Age'] <= 20,
                    'Revive',
                    'Decline'
                )
            )
        )
    )
)

# remove rows where period end year is before 2011
variable_df = variable_df[variable_df['Period End Year'] >= 2011]

variable_df.to_pickle('External/LSEG_variable_df_final.pkl')

Third, now we can join the LSEG data to the dataframe with the textual MAP measures.

In [ ]:
# Load the adjusted variable dataframe
variable_df = pd.read_pickle('External/LSEG_variable_df_final.pkl')


variable_df['Merge Key'] = variable_df['Instrument'] + '_' + variable_df['Period End Year'].astype(int).astype(str) + '_' + variable_df['Period End Month'].astype(int).astype(str)
# create a lagged merge key for merging the performance measures with 1 year lag (t+1), 2 years lag (t+2), 3 years lag (t+3), and 4 years lag (t+4)
variable_df['Merge Key lag 1'] = variable_df['Instrument'] + '_' + (variable_df['Period End Year'].astype(int) - 1).astype(str) + '_' + variable_df['Period End Month'].astype(int).astype(str)
variable_df['Merge Key lag 2'] = variable_df['Instrument'] + '_' + (variable_df['Period End Year'].astype(int) - 2).astype(str) + '_' + variable_df['Period End Month'].astype(int).astype(str)
variable_df['Merge Key lag 3'] = variable_df['Instrument'] + '_' + (variable_df['Period End Year'].astype(int) - 3).astype(str) + '_' + variable_df['Period End Month'].astype(int).astype(str)
variable_df['Merge Key lag 4'] = variable_df['Instrument'] + '_' + (variable_df['Period End Year'].astype(int) - 4).astype(str) + '_' + variable_df['Period End Month'].astype(int).astype(str)
variable_df['Merge Key lag 5'] = variable_df['Instrument'] + '_' + (variable_df['Period End Year'].astype(int) - 5).astype(str) + '_' + variable_df['Period End Month'].astype(int).astype(str)
variable_df['Merge Key lag 6'] = variable_df['Instrument'] + '_' + (variable_df['Period End Year'].astype(int) - 6).astype(str) + '_' + variable_df['Period End Month'].astype(int).astype(str)

corpus_df['Merge Key'] = corpus_df['Instrument'] + '_' + corpus_df['reporting_year'].astype(int).astype(str) + '_' + corpus_df['reporting_month'].astype(int).astype(str)

# merge the variable  (without duplicate columns) with the corpus dataframe

#if with_lag is True, we add 't+1' to the suffix of the variable columns to indicate that these are lagged variables,

if variable_df['Merge Key'].is_unique and corpus_df['Merge Key'].is_unique:
    merged_df = corpus_df.merge(variable_df,  on = 'Merge Key',  how = 'left', suffixes=('', '_y'))
else:
    print('Merge keys are not unique!')

del corpus_df

# Create new measures: R&D intensity as R&D expenses divided by total assets, and R&D ratio as R&D expenses divided by revenue,
# 'SGA_ratio' by dividing 'Selling General & Administrative Expenses - Total' by 'Operating Expenses - Total', and Inventory ratio as Inventory divided by total current assets
# 'Senior_Executive_Incentive' which is an indicator variable that is 1 if the companys' senior executive compensation is above the median, and 0 otherwise. 
# We will calculate the median senior executive compensation based on the merged dataframe, so that we only use the companies for which we have data on senior executive compensation to calculate the median.

merged_df['SGA_ratio'] = merged_df['Selling General & Administrative Expenses - Total'] / merged_df['Operating Expenses - Total']
merged_df['R&D Intensity'] = merged_df['Research & Development Expense'] / merged_df['Total Assets']
merged_df['R&D Ratio'] = merged_df['Research & Development Expense'] / merged_df['Revenue from Business Activities - Total']
merged_df['Inventory Ratio'] = merged_df['Inventories - Total'] / merged_df['Total Current Assets']
# Calculate median senior executive compensation
median_senior_executive_compensation = merged_df['Total Senior Executives Compensation'].median()
# Create indicator variable for senior executive incentive. If the senior executive compensation is missing, we set the indicator variable to NA
merged_df['Senior_Executive_Incentive'] = ((merged_df['Total Senior Executives Compensation'] > median_senior_executive_compensation) & (merged_df['Total Senior Executives Compensation'].notna())).astype(int)
merged_df['Senior_Executive_Incentive'] = [merged_df['Senior_Executive_Incentive'][i] if pd.notna(merged_df['Total Senior Executives Compensation'][i]) else None for i in range(len(merged_df))]
# Lastly, we join performance measures with 1 year lag (t+1) and 2 years lag (t+2) to the merged_df. We can do this because we have already created the merge keys for the lagged performance measures in the variable_df, and we have the same merge key in the merged_df.
# Before doing so, we rename 'Return on Average Total Assets - %, TTM' to ROA, 'EPS - Diluted - excl Exord Items Applicable to Common Total' to EPS, and 'Gross Profit Margin - %' to Gross_Margin in the variable_df for easier use in the merged_df
variable_df = variable_df.rename(columns={'Return on Average Total Assets - %, TTM': 'ROA_lag_t+1', 'EPS - Diluted - excl Exord Items Applicable to Common Total': 'EPS_lag_t+1', 'Gross Profit Margin - %': 'Gross_Margin_lag_t+1', 'Operating Margin - %': 'Operating_Margin_lag_t+1'})
# Now, we can join the lagged performance measures with the merged_df (do not take both merge keys into the merge).
merged_df = merged_df.merge(variable_df[['Merge Key lag 1', 'ROA_lag_t+1', 'EPS_lag_t+1', 'Gross_Margin_lag_t+1', 'Operating_Margin_lag_t+1']], left_on='Merge Key', right_on='Merge Key lag 1', how='left', suffixes=('', '_y'))
variable_df = variable_df.rename(columns={'ROA_lag_t+1': 'ROA_lag_t+2', 'EPS_lag_t+1': 'EPS_lag_t+2', 'Gross_Margin_lag_t+1': 'Gross_Margin_lag_t+2', 'Operating_Margin_lag_t+1': 'Operating_Margin_lag_t+2'})
merged_df = merged_df.merge(variable_df[['Merge Key lag 2', 'ROA_lag_t+2', 'EPS_lag_t+2', 'Gross_Margin_lag_t+2', 'Operating_Margin_lag_t+2']], left_on='Merge Key', right_on='Merge Key lag 2', how='left', suffixes=('', '_y'))
variable_df = variable_df.rename(columns={'ROA_lag_t+2': 'ROA_lag_t+3', 'EPS_lag_t+2': 'EPS_lag_t+3', 'Gross_Margin_lag_t+2': 'Gross_Margin_lag_t+3', 'Operating_Margin_lag_t+2': 'Operating_Margin_lag_t+3'})
merged_df = merged_df.merge(variable_df[['Merge Key lag 3', 'ROA_lag_t+3', 'EPS_lag_t+3', 'Gross_Margin_lag_t+3', 'Operating_Margin_lag_t+3']], left_on='Merge Key', right_on='Merge Key lag 3', how='left', suffixes=('', '_y'))
variable_df = variable_df.rename(columns={'ROA_lag_t+3': 'ROA_lag_t+4', 'EPS_lag_t+3': 'EPS_lag_t+4', 'Gross_Margin_lag_t+3': 'Gross_Margin_lag_t+4', 'Operating_Margin_lag_t+3': 'Operating_Margin_lag_t+4'})
merged_df = merged_df.merge(variable_df[['Merge Key lag 4', 'ROA_lag_t+4', 'EPS_lag_t+4', 'Gross_Margin_lag_t+4', 'Operating_Margin_lag_t+4']], left_on='Merge Key', right_on='Merge Key lag 4', how='left', suffixes=('', '_y'))
variable_df = variable_df.rename(columns={'ROA_lag_t+4': 'ROA_lag_t+5', 'EPS_lag_t+4': 'EPS_lag_t+5', 'Gross_Margin_lag_t+4': 'Gross_Margin_lag_t+5', 'Operating_Margin_lag_t+4': 'Operating_Margin_lag_t+5'})
merged_df = merged_df.merge(variable_df[['Merge Key lag 5', 'ROA_lag_t+5', 'EPS_lag_t+5', 'Gross_Margin_lag_t+5', 'Operating_Margin_lag_t+5']], left_on='Merge Key', right_on='Merge Key lag 5', how='left', suffixes=('', '_y'))
variable_df = variable_df.rename(columns={'ROA_lag_t+5': 'ROA_lag_t+6', 'EPS_lag_t+5': 'EPS_lag_t+6', 'Gross_Margin_lag_t+5': 'Gross_Margin_lag_t+6', 'Operating_Margin_lag_t+5': 'Operating_Margin_lag_t+6'})
merged_df = merged_df.merge(variable_df[['Merge Key lag 6', 'ROA_lag_t+6', 'EPS_lag_t+6', 'Gross_Margin_lag_t+6', 'Operating_Margin_lag_t+6']], left_on='Merge Key', right_on='Merge Key lag 6', how='left', suffixes=('', '_y'))

# drop the lagged merge keys from the merged_df as they are not needed anymore
merged_df = merged_df.drop(columns=['Merge Key lag 1', 'Merge Key lag 2', 'Merge Key lag 3', 'Merge Key lag 4', 'Merge Key lag 5', 'Merge Key lag 6', 'Merge Key lag 1_y', 'Merge Key lag 2_y', 'Merge Key lag 3_y', 'Merge Key lag 4_y', 'Merge Key lag 5_y', 'Merge Key lag 6_y'])

del variable_df

<div class='alert-info'>
Step 2: Retrieve monthly stock return data per company-month from LSEG.
</div>

First, we need to retrieve the monthly stock return data from the LSEG API. (We just need to do this once. For the other datasets use the final pickle file below)

In [ ]:
# Define the 'universe' of companies to get data from
universe = CIK_df['Instrument'].unique().astype(str).tolist()

#Open a new Refinitiv session (make sure that the desktop App is open)
rd.open_session()

# Updated fields list to include the date reference
monthly_return_df = rd.get_data(
    universe = universe,
    fields = ['TR.TotalReturn1Mo.date', 'TR.TotalReturn1Mo'], 
    parameters = {
        'SDate': '-181', # Start 180 months ago
        'EDate': '0',    # End at most recent
        'Frq': 'M',
        'CH': 'Id;Date', # Id for identifier, Date for the timestamp
        'RH': 'In'
    }
)

rd.close_session()

# Save the retrieved data to a pickle file
monthly_return_df.to_pickle('External/Monthly_Returns_with_Date.pkl')

Second, given the monthly stock return data, we create a new variable in the merged_df dataframe that entails the monthly stock return volatility for each company-fiscal year observation.

In [ ]:
# Load the monthly returns data with date reference
monthly_return_df = pd.read_pickle('External/Monthly_Returns_with_Date.pkl')

# remove rows with missing date or missing return
monthly_return_df = monthly_return_df[monthly_return_df['Date'].notna() & monthly_return_df['1 Month Total Return'].notna()]

# Given the data, we loop through the rows in merged_df and for each row (company-year), we calculate the Standard deviation of monthly stock returns for firm i in fiscal year t) 
# and save it in a new column 'Stock Return Volatility'. We use the date reference to filter the monthly returns for the respective fiscal year.
monthly_return_df['Date'] = pd.to_datetime(monthly_return_df['Date'])
monthly_return_df['Year'] = monthly_return_df['Date'].dt.year.astype(int)
monthly_return_df['Month'] = monthly_return_df['Date'].dt.month.astype(int)

# Now we can loop through the rows in merged_df and for each row, we filter the monthly returns for the respective company and fiscal year, 
# and calculate the standard deviation of the monthly returns to get the stock return volatility. We save this in a new column 'Stock Return Volatility' in merged_df.
merged_df['Stock Return Volatility'] = np.nan

for index, row in merged_df.iterrows():
    company_id = row['Instrument']
    fiscal_year = row['Period End Year']
    fiscal_month = row['Period End Month']
    # Filter the monthly returns for the respective company and fiscal year (go back 12 months from the fiscal year end date) 
    # For example if fiscal end is March 2020, we want to filter the monthly returns from April 2019 to March 2020. 
    # If fiscal end is December 2020, we want to filter the monthly returns from January 2020 to December 2020.
    if fiscal_month == 12:
        start_year = fiscal_year
        start_month = 1
    else:
        start_year = fiscal_year - 1
        start_month = fiscal_month + 1
    end_year = fiscal_year
    end_month = fiscal_month

    filtered_returns = monthly_return_df[
        (monthly_return_df['Instrument'] == company_id) &
        (
            ((monthly_return_df['Year'] == start_year) & (monthly_return_df['Month'] >= start_month)) |
            ((monthly_return_df['Year'] == end_year) & (monthly_return_df['Month'] <= end_month)) |
            ((monthly_return_df['Year'] > start_year) & (monthly_return_df['Year'] < end_year))
        )
    ]['1 Month Total Return']

    # Calculate the standard deviation of the monthly returns (if 12 data points are available) to get the stock return volatility and save it in the merged_df
    if len(filtered_returns) == 12:
        merged_df.loc[index, 'Stock Return Volatility'] = filtered_returns.std()
    else:
        merged_df.loc[index, 'Stock Return Volatility'] = np.nan

del monthly_return_df

<div class='alert-info'>
Step 3: Retrieve additional information on M&A deals per company-year from LSEG.
</div>

First, we need to retrieve the information on M&A deals per company-year from the LSEG API. (We just need to do this once. For the other datasets use the final pickle file below)

In [ ]:
#Define the 'universe' of companies to get data from
universe = CIK_df['Instrument'].unique()

# Define field list for M&A data retrieval
m_a_fields = ['TR.DealDate','TR.DealStatus','TR.DealAcquiror', 'TR.DealTarget', 'TR.DealType', 'TR.DealAttitude', 'TR.DealAnnouncementDate','TR.DealStartDate', 'TR.DealCloseDate', 'TR.DealClosedDate']

#Open a new Refinitiv session (make sure that the desktop App is open)
rd.open_session()

#Get M&A variables from Refinitiv for all companies in the universe for the past 14 years
deal_df = rd.get_data(
    universe = universe,
    fields = m_a_fields,
    parameters = {
        'SDate': '-15Y', # Look back 15 years
        'EDate': '0'
    }
)

rd.close_session()

# Retrieve the Announcement Year from the Deal Date (in df as 'Date') as integer variable
deal_df['Announcement_Year'] = pd.to_datetime(deal_df['Date']).dt.year.astype('Int64')

# Retrieve the Closed Year from the Deal Date (in df as 'Date') as integer variable
deal_df['Closed_Year'] = pd.to_datetime(deal_df['Closed Date']).dt.year.astype('Int64')

# Filter for completed deals and M&A deals only (no Investments, Divestitures, etc.)
deal_df = deal_df[(deal_df['Deal Status'] == 'Complete') & (deal_df['Deal Type'] == 'Mergers & Acquisitions')]

# Save the M&A variable dataframe
deal_df.to_pickle('External/M&A_df_final_new.pkl')

Second, based on the retrived data, we create some new M&A variables and join them to the main dataframe.

In [ ]:
# new merge key for M&A deals
merged_df['Merge Key 2'] = merged_df['Instrument'] + '_' + merged_df['reporting_year'].astype(int).astype(str)

# Include number of M&A deals per fiscal year and an indicator variable for whether there was at least one M&A deal in the fiscal year
deal_df = pd.read_excel('Tables_new/M&A_df_final_new.xlsx')
deal_counts = deal_df.groupby(['Instrument', 'Closed_Year']).size().reset_index(name='Num_MnA_Deals')

# Create a merge key for joining with the main dataframe
deal_counts['Merge Key 2'] = deal_counts['Instrument'] + '_' + deal_counts['Closed_Year'].astype(int).astype(str)

# Merge the deal counts with the merged dataframe (if no deals, fill with 0)
merged_df = merged_df.merge(deal_counts[['Merge Key 2', 'Num_MnA_Deals']], on='Merge Key 2', how='left')

merged_df['Num_MnA_Deals'] = merged_df['Num_MnA_Deals'].fillna(0).astype(int)

merged_df['MnA_Deal_Indicator'] = np.where(merged_df['Num_MnA_Deals'] > 0, 1, 0)

del deal_df, deal_counts

<div class='alert-info'>
Step 4: Retrieve information on the ownership structure per company-year from LSEG.
</div>

First, we need to retrieve the ownership structure data (top 20 investors by holding percentage) from the LSEG API. (We just need to do this once. For the other datasets use the final pickle file below)

NOTE: This part may take a while. It might be possible that the Refinitiv API will not return all data in one go, so we will have to run this part multiple times to get all the data. We will save the data after each run and then merge it with the previous runs.

In [ ]:
# Define the 'universe' of companies to get data from
universe = CIK_df['Instrument'].unique()
# Define field list for M&A data retrieval
ownership_fields = ['TR.InvestorFullName', 'TR.PctOfSharesOutHeld', 'TR.InvParentType', 'TR.InvestorType', 'TR.SharesHeld', 'TR.SharesHeldValue', 'TR.HoldingsDate']

# Retrieve the already processed data, or create empty dataframe if not existing
if os.path.exists('External/Ownership_top_20_investors.pkl'):
    ownership_df = pd.read_pickle('External/Ownership_top_20_investors.pkl')
else:
    ownership_df = pd.DataFrame()

# Exclude already processed instruments from the universe to avoid redundant data retrieval
if not ownership_df.empty:
    processed_instruments = ownership_df['Instrument'].unique()
    universe = [inst for inst in universe if inst not in processed_instruments]

#Open a new Refinitiv session (make sure that the desktop App is open)
rd.open_session()

#Get M&A variables from Refinitiv for all companies in the universe for the past 14 years (just top 20 investors)
i=0
for instrument in universe:
    print(f'Processing instrument: {instrument}')
    i+=1

    # retrieve ownership data for each instrument and the respective year
    ownership_tmp_df = rd.get_data(
        universe = instrument,
        fields = ownership_fields,
        parameters = {
                'SDate': f'-1',
                'EDate': f'-14',
                'Frq': 'FY',
                'CH': 'IN;Fd',
                'RH': 'date'
            }
        )

    #Convert 'Holdings Pct Of Traded Shares Held' to numeric
    ownership_tmp_df['Holdings Pct Of Traded Shares Held'] = pd.to_numeric(ownership_tmp_df['Holdings Pct Of Traded Shares Held'], errors='coerce')

    # Sort and take top 20 for each group (year)
    ownership_tmp_df['Holding Year'] = pd.to_datetime(ownership_tmp_df['Holdings Filing Date'], errors='coerce').dt.year.astype('Int64')

    ownership_tmp_df = ownership_tmp_df.sort_values(['Holding Year', 'Holdings Pct Of Traded Shares Held'], ascending=[False, False], na_position='last').groupby('Holding Year').head(20)

    # extract the top 20 firms and add them to the main top 20 dataframe
    if ownership_df.empty:
        ownership_df = ownership_tmp_df.reset_index(drop=True)
    else:
        ownership_df = pd.concat([ownership_df, ownership_tmp_df], ignore_index=True).reset_index(drop=True)

    # save intermediate results every 5 instruments
    if i % 5 == 0:
        ownership_df.to_pickle(f'External/Ownership_top_20_investors.pkl')

rd.close_session()

del ownership_fields, universe, instrument, ownership_tmp_df

ownership_df.to_pickle(f'External/Ownership_top_20_investors.pkl')

Second, once the ownership data is retrieved, we create some new measures.

In [ ]:
# Once the download of the ownership data for the top 20 investors per year is complete, we can now create new measures based on the ownership data at the firm-year level
ownership_df = pd.read_pickle(f'External/Ownership_top_20_investors.pkl')

# First, we need to create a new dataframe that contains one row per firm-year level. As of now just the columns 'Instrument' and 'Holding Year'
ownership_measures_df = ownership_df[['Instrument', 'Holding Year']].drop_duplicates().reset_index(drop=True)

# Second, we create a new measures that indicates the shareholding of the largest shareholder (in terms of percentage of shares held) at the firm-year level.
largest_shareholder = ownership_df.groupby(['Instrument', 'Holding Year'])['Holdings Pct Of Traded Shares Held'].max()/100

ownership_measures_df['Largest Shareholder'] = ownership_measures_df.set_index(['Instrument', 'Holding Year']).index.map(largest_shareholder)

# Third, we create measure for the ownership concentration of the top 10 institutional investors at the firm-year level by summing up the squared percentage of shares held by the top 10 institutional investors.
non_institutional = ['Individual Investor', 'Independent Research Firm', 'Research Firm', 'Family Office', 'Corporation', 'Holding Company', 'Other Insider Investor']
institutional = [inst for inst in ownership_df['Investor Type Description'].unique() if inst not in non_institutional]

ownership_df['Is Institutional'] = ownership_df['Investor Type Description'].apply(lambda x: 1 if x in institutional else 0)

top_10_institutional = ownership_df[ownership_df['Is Institutional'] == 1].groupby(['Instrument', 'Holding Year'])['Holdings Pct Of Traded Shares Held'].nlargest(10)

ownership_measures_df['Top 10 Institutional Concentration'] = ownership_measures_df.set_index(['Instrument', 'Holding Year']).index.map(top_10_institutional.groupby(level=[0,1]).apply(lambda x: np.sum(np.square(x/100)) if len(x) > 0 else np.nan))

# Save the ownership measures dataframe
ownership_measures_df.to_pickle(f'External/Ownership_measures_final.pkl')

del ownership_df, ownership_measures_df, largest_shareholder, top_10_institutional, non_institutional, institutional


Third, we can join the ownership measures to the main data set.

In [ ]:
# merge the ownership measures with the main merged dataframe
ownership_measures_df = pd.read_pickle(f'External/Ownership_measures_final.pkl')

ownership_measures_df['Merge Key 2'] = ownership_measures_df['Instrument'] + '_' + ownership_measures_df['Holding Year'].astype(int).astype(str)
merged_df = merged_df.merge(ownership_measures_df[['Merge Key 2', 'Largest Shareholder', 'Top 10 Institutional Concentration']], on='Merge Key 2', how='left')

del ownership_measures_df

<div class='alert-info'>
Step 5: Load executive (here: mainly CEO) compensation data from the ISS Incentive Lab data set.
</div>

First, we need to join the variables from the two tables 'ParticipantFY' and 'SumComp'. In the presented case we downloaded the U.S. dataset (version: 20250201) and move the raw tables 'ParticipantFY.csv' and 'SumComp.csv' into the 'External' folder. Afterwards, we join the tables as follow:

In [ ]:
# Load the raw tables 'ParticipantFY.csv' and 'SumComp.csv' 
participant_df = pd.read_csv('External/ParticipantFY.csv', sep=',')
sumcomp_df = pd.read_csv('External/SumComp.csv', sep=',')

# Change the variable type of 'CIK' to to string in both dataframes (and remove "'")
participant_df['CIK'] = participant_df['CIK'].str.replace("'", "").astype(str)
sumcomp_df['CIK'] = sumcomp_df['CIK'].str.replace("'", "").astype(str)

# Create a merge key based on Participant ID ('participantid') and fiscal year ('FiscalYear') and fiscal month ('FiscalMonth') in both dataframes
participant_df['Merge Key'] = participant_df['participantid'].astype(str) + '_' + participant_df['FiscalYear'].astype(int).astype(str) + '_' + participant_df['FiscalMonth'].astype(int).astype(str)
sumcomp_df['Merge Key'] = sumcomp_df['participantid'].astype(str) + '_' + sumcomp_df['FiscalYear'].astype(int).astype(str) + '_' + sumcomp_df['FiscalMonth'].astype(int).astype(str)

# Remove duplicate rows in both dataframes based on the merge key
participant_df = participant_df.drop_duplicates(subset=['Merge Key'], keep='first')
sumcomp_df = sumcomp_df.drop_duplicates(subset=['Merge Key'], keep='first')

# Filter participant_df to only include rows with currentCEO == 1 and FiscalYear 2012 - 2024
participant_df = participant_df[(participant_df['currentCEO'] == 1) & (participant_df['FiscalYear'].between(2012, 2024))]

# Filter sumcomp_df to only include rows with FiscalYear 2012 - 2024. 
# And just keep the columns 'Merge Key', 'salary', 'bonus', 'stockAwards', 'optionAwards', 'nonEquityComp', 'pensionNQDC', 'otherComp', and 'totalComp'
sumcomp_df = sumcomp_df[sumcomp_df['FiscalYear'].between(2012, 2024)]
sumcomp_df = sumcomp_df[['Merge Key', 'salary', 'bonus', 'stockAwards', 'optionAwards', 'nonEquityComp', 'pensionNQDC', 'otherComp', 'totalComp']]

# Merge the participant_df and sumcomp_df dataframes (inner join) on the merge key to get the CEO compensation data
ceo_comp_df = participant_df.merge(sumcomp_df, on='Merge Key', how='inner')

# Save the CEO compensation dataframe to a pickle file
ceo_comp_df.to_pickle('External/ISS_CEO_Compensation_final.pkl')

Second, based on the information provided for the cash and equity compensation, we create some new variables and merge them to the main dataframe.

In [ ]:
# merge the ISS measures with the main merged dataframe
comp_measures_df = pd.read_pickle(f'External/ISS_CEO_Compensation_final.pkl')

#make CIK code 10 digits with leading zeros
comp_measures_df['CIK'] = comp_measures_df['CIK'].apply(lambda x: str(x).zfill(10))

# merge Instrument codes to comps dataset
comp_measures_df = pd.merge(comp_measures_df, CIK_df[['CIK Number', 'Instrument']], left_on='CIK', right_on='CIK Number', how='left')

# create merge key for merging with main dataframe
comp_measures_df['Merge Key'] = comp_measures_df['Instrument'] + '_' + comp_measures_df['FiscalYear'].astype(int).astype(str) + '_' + comp_measures_df['FiscalMonth'].astype(int).astype(str)
# drop duplicate rows based on the merge key, keeping the first occurrence
comp_measures_df = comp_measures_df.drop_duplicates(subset=['Merge Key'], keep='first')
# Now we can merge the comp measures with the main merged dataframe using the merge key. We only merge the relevant columns for the CEO compensation measures, which are 'salary', 'bonus', 'stockAwards', 'optionAwards', 'nonEquityComp', 'pensionNQDC', 'otherComp', and 'totalComp'.
merged_df = merged_df.merge(comp_measures_df[['Merge Key', 'salary', 'bonus', 'stockAwards', 'optionAwards', 'nonEquityComp', 'pensionNQDC', 'otherComp', 'totalComp']], on='Merge Key', how='left')
# Create new measures for CEO cash compensation and CEO cash incentive based on the salary, bonus, and non-equity compensation variables. 
# We define CEO cash compensation as the sum of salary, bonus, and non-equity compensation, 
# and CEO cash incentive as an indicator variable that is 1 if the CEO cash compensation is above the median CEO cash compensation in the sample, and 0 otherwise. 
# If the CEO cash compensation is missing, we set the CEO cash incentive variable to NA.
merged_df['CEO_Cash_Compensation'] = merged_df['salary'] + merged_df['bonus'] + merged_df['nonEquityComp']
merged_df['CEO_Cash_Incentive'] = np.where(merged_df['CEO_Cash_Compensation'] > merged_df['CEO_Cash_Compensation'].median(), 1, 0)
merged_df['CEO_Cash_Incentive'] = [merged_df['CEO_Cash_Incentive'][i] if pd.notna(merged_df['CEO_Cash_Compensation'][i]) else None for i in range(len(merged_df))]
merged_df['CEO_Equity_Compensation'] = merged_df['stockAwards'] + merged_df['optionAwards']
merged_df['CEO_Equity_Incentive'] = np.where(merged_df['CEO_Equity_Compensation'] > merged_df['CEO_Equity_Compensation'].median(), 1, 0)
merged_df['CEO_Equity_Incentive'] = [merged_df['CEO_Equity_Incentive'][i] if pd.notna(merged_df['CEO_Equity_Compensation'][i]) else None for i in range(len(merged_df))]

del comp_measures_df

<div class='alert-info'>
Step 6: Load patent data from the DISCERN Database.
</div>

First, we need to retrieve the DISCERN dataset first from the official website: https://zenodo.org/records/13619821. In the presented case we downloaded the version 2.0.1 and move the raw tables 'discern_firm_panel_1980_2021.csv' and 'Patent_Panel_Data_2015.csv' into the 'External' folder.

In [ ]:
# Start with 'discern_firm_panel_1980_2021.csv' and make a left outer join with 'Patent_Panel_Data_2015.csv' based on the 'gvkey'. From the table 'Patent_Panel_Data_2015.csv' we will just join the variables 'conm' and 'pat_yr'
discern_df = pd.read_csv('External/discern_firm_panel_1980_2021.csv', sep=',')
patent_df = pd.read_csv('External/Patent_Panel_Data_2015.csv', sep=',')

# Drop duplicate rows in patent_df based on the 'gvkey' variable, keeping the first occurrence of each gvkey. We will keep only the columns 'gvkey', 'cusip', and 'conm' from the patent_df for merging with the discern_df.
patent_df = patent_df[['gvkey', 'cusip', 'conm']].drop_duplicates(subset=['gvkey'], keep='first')

# Merge the two dataframes based on the 'gvkey' variable, keeping all rows from the discern_df and only matching rows from the patent_df
merged_patent_df = pd.merge(discern_df, patent_df, on='gvkey', how='left')

del discern_df, patent_df

# Rename the column 'cusip' to 'CUSIP' and remove error or missings from CUSIP column (i.e., remove rows where CUSIP is missing or empty string)
merged_patent_df = merged_patent_df.rename(columns={'cusip': 'CUSIP'})
merged_patent_df = merged_patent_df[merged_patent_df['CUSIP'].notna() & (merged_patent_df['CUSIP'] != '')]

# Just keep observations with 'fyear' between 2012 and 2021 (inclusive) and drop the rest
merged_patent_df = merged_patent_df[(merged_patent_df['fyear'] >= 2012) & (merged_patent_df['fyear'] <= 2021)]

# Save the merged dataframe to a pickle file
merged_patent_df.to_pickle('External/DISCERN_2012_2021.pkl')

del merged_patent_df

Second, we create the inovation variables and join them to the main dataframe.

In [ ]:
# Load DISCERN DATA 
discern_df = pd.read_pickle('External/DISCERN_2012_2021.pkl')

# Create new Innovation Variable by taken pat_grant_fyear and pat_app_fyear and take log of it (add 1 to avoid log of 0)
discern_df['log_grant_pat'] = np.log(discern_df['pat_grant_fyear'] + 1)

discern_df['log_app_pat'] = np.log(discern_df['pat_app_fyear'] + 1)

merged_df['Merge Key 3'] = merged_df['CUSIP'] + '_' + merged_df['reporting_year'].astype(int).astype(str)

# Merge Key is CUSIP_YEAR for merging with main dataframe
discern_df['Merge Key 3'] = discern_df['CUSIP'] + '_' + discern_df['fyear'].astype(int).astype(str)
merged_df = merged_df.merge(discern_df[['Merge Key 3', 'pat_grant_fyear', 'pat_app_fyear', 'log_grant_pat', 'log_app_pat']], on='Merge Key 3', how='left')

del discern_df

# Drop duplicate observations in the merged_df based on the 'Merge Key' variable, keeping the first occurrence of each Merge Key
merged_df = merged_df.drop_duplicates(subset=['Merge Key'], keep='first')

# Save the final merged dataframe with all variables for the validation analyses
if within_industry:
    merged_df.to_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_within_industry_normalized.pkl')
elif without_finance:
    merged_df.to_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_without_finance_normalized.pkl')
else:
    merged_df.to_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_final.pkl')


<div class='alert-info'>
Step 7 (After all Merged files are created): Make sure that all files contain the same firm-year observations.
</div>

In [ ]:
# Specify whether to use MAP measures normalized across the whole sample (False) or within each industry (True)
within_industry = False  

# Specify whether to exclude firms operating in the Financing/Investment sector (true or false).
without_finance = False

# Now, we want to make sure that we have the same sample for the regression analysis of the different measurement types for a better comparability of the results across the different measurement types.
measurement_types = ['BoW', 'W2V_v1', 'W2V_v2']

# Load the merged dataframe for the GLLM corpus, which we will use as a reference to identify the common merge keys between the dictionary and GLLM corpora
if within_industry:
    merged_df_gllm = pd.read_pickle('GLLM/Validation_Merge_GLLM_within_industry_normalized.pkl')
elif without_finance:
    merged_df_gllm = pd.read_pickle('GLLM/Validation_Merge_GLLM_without_finance_normalized.pkl')
else:
    merged_df_gllm = pd.read_pickle('GLLM/Validation_Merge_GLLM_final.pkl')

for measurement_type in measurement_types:
    # load the merged dataframe for the respective measurement type
    if within_industry:
        merged_df = pd.read_pickle(f'{measurement_type.split("_")[0]}/Validation_Merge_{measurement_type}_within_industry_normalized.pkl')
    elif without_finance:
        merged_df = pd.read_pickle(f'{measurement_type.split("_")[0]}/Validation_Merge_{measurement_type}_without_finance_normalized.pkl')
    else:
        merged_df = pd.read_pickle(f'{measurement_type.split("_")[0]}/Validation_Merge_{measurement_type}_final.pkl')
    # get the unique merge keys for the respective measurement type and the GLLM merged dataframe
    merge_key_2_dictionary = merged_df['Merge Key 2'].unique()
    merge_key_2_gllm = merged_df_gllm['Merge Key 2'].unique()
    # identify the common merge keys between the W2V and GLLM corpora
    common_merge_keys = set(merge_key_2_dictionary).intersection(set(merge_key_2_gllm))

    # number of common merge keys between the W2V and GLLM corpora
    print(f'Number of common merge keys between {measurement_type} and GLLM corpora: {len(common_merge_keys)}')

    # print those that are only in the W2V corpus and not in the GLLM corpus
    only_in_w2v = set(merge_key_2_dictionary) - set(merge_key_2_gllm)
    print(f'Number of merge keys only in {measurement_type} corpus: {len(only_in_w2v)}')

    # print those that are only in the GLLM corpus and not in the W2V corpus
    only_in_gllm = set(merge_key_2_gllm) - set(merge_key_2_dictionary)
    print(f'Number of merge keys only in {measurement_type} corpus: {len(only_in_gllm)}')

    # drop rows that are only in the W2V corpus and not in the GLLM corpus or only in the GLLM corpus and not in the W2V corpus to make sure that we have the same sample for the regression analysis of the different measurement types for a better comparability of the results across the different measurement types.
    merged_df = merged_df[merged_df['Merge Key 2'].isin(common_merge_keys)].reset_index(drop=True)

    # save the merged dataframe with only the common merge keys for the regression analysis of the different measurement types for a better comparability of the results across the different measurement types.
    if within_industry:
        merged_df.to_pickle(f'{measurement_type.split("_")[0]}/Validation_Merge_{measurement_type}_within_industry_normalized.pkl')
    elif without_finance:
        merged_df.to_pickle(f'{measurement_type.split("_")[0]}/Validation_Merge_{measurement_type}_without_finance_normalized.pkl')
    else:
        merged_df.to_pickle(f'{measurement_type.split("_")[0]}/Validation_Merge_{measurement_type}_final.pkl')

del merged_df, merged_df_gllm, merge_key_2_dictionary, merge_key_2_gllm, common_merge_keys, only_in_w2v, only_in_gllm


## Concurrent/Convergent Validity

Load the merged corpus with the normalized MAP measures. We can choose between the different measurement approaches and normalization procedures by changing the 'measurement_approach', 'wihtin_industry', and 'without_finance' variables below. 

NOTE: In the paper just the baseline configuration (within_industry = False, without_finance = True) is shown. But the same can be done with the other possible settings.

In [ ]:
# Load the merged corpus with the normalized MAP measures and the additional variables
# We can choose between the different measurement types by changing the 'measurement_approach', 'within_industry' and 'without_finance' variables below.

# Specifiy the measurement approach to use for the analysis. Options are: 'BoW', 'W2V_v1', 'W2V_v2', 'GLLM'
measurement_approach = 'GLLM'

# Specify whether to use MAP measures normalized across the whole sample (False) or within each industry (True)
within_industry = False  

# Specify whether to exclude firms operating in the Financing/Investment sector (true or false).
without_finance = True

# Load the merged dataframe for the respective measurement type
if within_industry:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_within_industry_normalized.pkl')
elif without_finance:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_without_finance_normalized.pkl')
else:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_final.pkl')

# drop duplicates in column 'Merge Key' if there are any (we just want one observation per firm-year for the upcoming analyses)
if merged_df['Merge Key'].duplicated().sum() > 0:
    merged_df = merged_df.drop_duplicates(subset=['Merge Key'], keep='last').reset_index(drop=True)

Next we remove columns that are not relevant for the concurrent/convergent analyses (e.g., identifiers, non-normalized measures, etc.). The relevant columns are the normalized MAP dimension measures (e.g., tf-idf weighted measures, equally weighted measures, etc.) and the financial, ESG, and additional variables that we want to use for the validation analyses.

In [ ]:
# First, we create a dataframe with all available columns
relevant_columns = merged_df.columns.tolist()

# remove further columns that are not relevant for the random forrest analysis (e.g., identifiers, non-normalized measures, etc.)
columns_to_remove = ['Merge Key', 'Date of Incorporation', 'IPO Date', 'Period End Date', 'Period End Month', 'Year of Incorporation', 'IPO Year', 'MAP_token_tf_idf',
                      'Budgeting_Planning_tf_idf_weighted', 'Cost_tf_idf_weighted', 'Financing_Investment_tf_idf_weighted', 'Operations_tf_idf_weighted', 'Performance_Internal_Reporting_tf_idf_weighted',
                      'Pricing_Revenue_Management_tf_idf_weighted', 'Risk_Internal_Control_tf_idf_weighted', 'Strategy_tf_idf_weighted', 'Instrument_y', 'MAP_token_count','Budget_equally_weighted',
                      'Budgeting_Planning_equally_weighted', 'Cost_equally_weighted', 'Financing_Investment_equally_weighted', 'Operations_equally_weighted', 'Performance_Internal_Reporting_equally_weighted',
                      'Pricing_Revenue_Management_equally_weighted', 'Risk_Internal_Control_equally_weighted', 'Strategy_equally_weighted', 'NAICS Subsector Name', 'NAICS Industry Group Name', 'Budget_tf_idf_weighted_scaled',
                      'Budgeting_Planning_tf_idf_weighted_scaled', 'Cost_tf_idf_weighted_scaled', 'Financing_Investment_tf_idf_weighted_scaled', 'Operations_tf_idf_weighted_scaled', 'Performance_Internal_Reporting_tf_idf_weighted_scaled',
                      'Pricing_Revenue_Management_tf_idf_weighted_scaled', 'Risk_Internal_Control_tf_idf_weighted_scaled', 'Strategy_tf_idf_weighted_scaled', 'Budget_equally_weighted_scaled',
                      'Budgeting_Planning_equally_weighted_scaled', 'Cost_equally_weighted_scaled', 'Financing_Investment_equally_weighted_scaled', 'Operations_equally_weighted_scaled', 'Performance_Internal_Reporting_equally_weighted_scaled',
                      'Pricing_Revenue_Management_equally_weighted_scaled', 'Risk_Internal_Control_equally_weighted_scaled', 'Strategy_equally_weighted_scaled', 'NAICS Subsector All Code',
                      'Investment_equally_weighted', 'Investment_tf_idf_weighted_scaled', 'Investment_equally_weighted_scaled', 'Performance_equally_weighted', 'Performance_tf_idf_weighted_scaled', 'Performance_equally_weighted_scaled',
                      'Risk_equally_weighted', 'Risk_tf_idf_weighted_scaled', 'Risk_equally_weighted_scaled', 'Budget_tf_idf_weighted', 'Investment_tf_idf_weighted', 'Performance_tf_idf_weighted', 'Risk_tf_idf_weighted',
                      'file_name', 'reporting_year', 'reporting_month', 'filing_year', 'CIK', 'filing_text', 'section_error', 'filing_key', 'company', 'Instrument_x', 'accession_number', 'Word_count', 'Company Name', 
                      'words_in_sentences_less_than_5_words', 'words_in_sentences_more_than_800_words', 'share_words_in_sentences_less_than_5_words', 'share_words_in_sentences_more_than_800_words',
                      'num_sentences_less_than_5_words', 'num_sentences_more_than_800_words', 'max_words_per_sentence', 'num_entries', 'results', 'prompting_time_minutes', 'results_FT', 'prompting_time_minutes_FT', 
                      'Cost_df', 'Cost_FT_df', 'Investment_df', 'Investment_FT_df', 'Operations_df', 'Operations_FT_df', 'Performance_df', 'Performance_FT_df', 'Risk_df', 'Risk_FT_df', 'Strategy_df', 'Strategy_FT_df', 
                        'Pricing_df', 'Pricing_FT_df', 'Budget_df', 'Budget_FT_df', 'Cost_explicit_count', 'Cost_implicit_count', 'Cost_explicit_count_FT', 'Cost_implicit_count_FT', 'Financing_Investment_explicit_count', 
                        'Financing_Investment_implicit_count', 'Financing_Investment_explicit_count_FT', 'Financing_Investment_implicit_count_FT', 'Operations_explicit_count', 'Operations_implicit_count', 
                        'Operations_explicit_count_FT', 'Operations_implicit_count_FT', 'Performance_Internal_Reporting_explicit_count', 'Performance_Internal_Reporting_implicit_count', 
                        'Performance_Internal_Reporting_explicit_count_FT', 'Performance_Internal_Reporting_implicit_count_FT', 'Risk_Internal_Control_explicit_count', 'Risk_Internal_Control_implicit_count', 
                        'Risk_Internal_Control_explicit_count_FT', 'Risk_Internal_Control_implicit_count_FT', 'Strategy_explicit_count', 'Strategy_implicit_count', 'Strategy_explicit_count_FT', 
                        'Strategy_implicit_count_FT', 'Pricing_Revenue_Management_explicit_count', 'Pricing_Revenue_Management_implicit_count', 'Pricing_Revenue_Management_explicit_count_FT', 
                        'Pricing_Revenue_Management_implicit_count_FT', 'Budgeting_Planning_explicit_count', 'Budgeting_Planning_implicit_count', 'Budgeting_Planning_explicit_count_FT', 
                        'Budgeting_Planning_implicit_count_FT', 'Cost_explicit_CS_count', 'Cost_implicit_CS_count', 'Cost_explicit_CS_count_FT', 'Cost_implicit_CS_count_FT', 
                        'Financing_Investment_explicit_CS_count', 'Financing_Investment_implicit_CS_count', 'Financing_Investment_explicit_CS_count_FT', 'Financing_Investment_implicit_CS_count_FT', 
                        'Operations_explicit_CS_count', 'Operations_implicit_CS_count', 'Operations_explicit_CS_count_FT', 'Operations_implicit_CS_count_FT', 'Performance_Internal_Reporting_explicit_CS_count', 
                        'Performance_Internal_Reporting_implicit_CS_count', 'Performance_Internal_Reporting_explicit_CS_count_FT', 'Performance_Internal_Reporting_implicit_CS_count_FT', 
                        'Risk_Internal_Control_explicit_CS_count', 'Risk_Internal_Control_implicit_CS_count', 'Risk_Internal_Control_explicit_CS_count_FT', 'Risk_Internal_Control_implicit_CS_count_FT', 
                        'Strategy_explicit_CS_count', 'Strategy_implicit_CS_count', 'Strategy_explicit_CS_count_FT', 'Strategy_implicit_CS_count_FT', 'Pricing_Revenue_Management_explicit_CS_count', 
                        'Pricing_Revenue_Management_implicit_CS_count', 'Pricing_Revenue_Management_explicit_CS_count_FT', 'Pricing_Revenue_Management_implicit_CS_count_FT', 'Budgeting_Planning_explicit_CS_count', 
                        'Budgeting_Planning_implicit_CS_count', 'Budgeting_Planning_explicit_CS_count_FT', 'Budgeting_Planning_implicit_CS_count_FT', 'Merge Key 2', 'Merge Key 3', 
                        'Life Cycle Stage 1', 'Life Cycle Stage 2', 'CUSIP']

# Remove the columns that are not relevant for the random forest analysis
relevant_columns = [col for col in relevant_columns if col not in columns_to_remove]

# Next, we create a new dataframe that only contains the relevant columns for the random forest analysis
merged_df = merged_df[relevant_columns]

# Remove rows with missing values in the relevant control variables columns for the regression analyses for the convergent validity checks (i.e., 'Total Assets' and 'Return on Average Total Assets - %, TTM')
merged_df = merged_df.dropna(subset=['Instrument', 'Period End Year', 'NAICS Sector Name', 'Total Assets', 'Return on Average Total Assets - %, TTM'])

# Finally, we can also create dummy variables for the categorical variables (e.g., NAICS Sector Name) if needed for the random forest analysis.
merged_df_encoded = pd.get_dummies(
    merged_df[relevant_columns],
    columns=['NAICS Sector Name'],
    drop_first=True
)

del columns_to_remove, relevant_columns

Depending on the MAP dimension under investigation we consider different variables for the concurrent/convergent validity tests. These variables are first chosen to be as close as possible to the variable picked by [Qiu et al. (2023)](https://www.sciencedirect.com/science/article/pii/S1044500522000361) for their validity analyses, but we also include additional variables that we deem relevant for the respective dimension based on the available data.

In [ ]:
# Define the variables for each MAP dimension for the random forest analysis. We will use these variables to create a new dataframe that only contains the relevant columns for the random forest analysis.
Budgeting_Planning_columns = ['Working Capital to Total Assets', 'Management Departures', 'Succession Plan', 
'Total Debt Percentage of Total Assets', 'Payables Turnover', 'Accounts Receivable Turnover', 'Average Payables Payment Days', 'Inventory Turnover',
'Announced Layoffs To Total Employees', 'Earnings Retention Rate', 'Average Net Trade Cycle Days', 'Total Debt Percentage of Total Equity',
'Average Receivables Collection Days', 'Crisis Management Systems', 'Reinvestment Rate - %, TTM', 'Inventory Ratio']

Cost_columns = ['Asset Turnover', 'Average Inventory Days', 'SGA_ratio', 'Current Ratio', 'Inventory Turnover', 'Average Payables Payment Days', 
'Earnings Retention Rate', 'Announced Layoffs To Total Employees', 'Average Receivables Collection Days', 
'Interest Coverage Ratio', 'Payables Turnover', 'Accounts Receivable Turnover', 'Cost of Operating Revenue', 'Product Recall', 'Inventory Ratio']

Financing_Investment_columns = ['Total Debt Percentage of Total Assets', 'Interest Coverage Ratio', 'Net Cash Flow from Financing Activities',
 'Dividend Yield - Common - Net - Issue - %, TTM', 'Net Cash Flow from Investing Activities', 'Average Net Trade Cycle Days', 'Reinvestment Rate - %, TTM',
'Total Debt Percentage of Total Equity', 'EPS - Diluted - excl Exord Items Applicable to Common Total', 'Average Payables Payment Days', 
'Free Cash Flow Yield - %, TTM', 'Earnings Retention Rate', 'MnA_Deal_Indicator', 'Return on Invested Capital - %, TTM', 'Payables Turnover',
'Num_MnA_Deals', 'Property Plant & Equipment - Net - Total',  'Current Ratio'] 

Operations_columns = ['Training and Development Policy', 'Product Recall', 'Inventory Turnover',  
'Product Quality Controversies Score', 'Six Sigma and Quality Mgt Systems', 'ISO 9000', 'Average Inventory Days', 
'Cost of Operating Revenue', 'Management Training', 'Product Responsibility Score', 'Inventory Ratio'] 

Performance_Internal_Reporting_columns = ['Management Training', 'Training and Development Policy', 
'Total Senior Executives Compensation', 'Senior_Executive_Incentive', 'Product Responsibility Monitoring', 'Crisis Management Systems', 
'CEO_Cash_Compensation', 'CEO_Cash_Incentive', 'CEO_Equity_Compensation', 'CEO_Equity_Incentive'] 

Pricing_Revenue_Management_columns = [ 'Product Responsibility Score', 'Product Responsibility Monitoring', 'Product Recall', 
'Sales per Employee', 'Average Receivables Collection Days', 'Accounts Receivable Turnover', 'Asset Turnover', 'Average Inventory Days',
'Inventory Turnover', 'Net Change in Cash - Total', 'Free Cash Flow Yield - %, TTM', 'Cash & Cash Equivalents', 
'Net Cash Flow from Operating Activities', 'Product Quality Controversies Score'] 

Risk_Internal_Control_columns = ['Internal Audit Department Reporting Score', 'Audit Committee Expertise Score', 'Crisis Management Systems', 
'Product Responsibility Score', 'Management Score', 'Succession Plan', 'Product Responsibility Monitoring', 
'Total Debt Percentage of Total Assets', 'Total Debt Percentage of Total Equity', 'ESG Controversies Score'] 

Strategy_columns = ['MnA_Deal_Indicator', 'Num_MnA_Deals', 'Net Cash Flow from Investing Activities', 
'Acquisition & Disposal of Business Sold/(Acquired) Net - CF', 'EPS - Diluted - excl Exord Items Applicable to Common Total',
'Management Training', 'Integrated Strategy in MD&A Score', 'Training and Development Policy', 'Reinvestment Rate - %, TTM', 'Management Score', 
'pat_grant_fyear', 'pat_app_fyear', 'R&D Ratio'] 

Industry_columns = merged_df_encoded.filter(like='NAICS Sector Name_').columns.tolist()

# Depending on the measurement type, we have different MAP dimensions and respective variables that we include in the analysis
if measurement_approach == 'BoW':
    MAP_dims = ['Budget', 'Cost', 'Investment', 'Operations', 'Performance', 'Risk', 'Strategy']
else:
    MAP_dims = ['Budgeting_Planning', 'Cost', 'Financing_Investment', 'Operations', 'Performance_Internal_Reporting', 'Pricing_Revenue_Management', 'Risk_Internal_Control', 'Strategy']

# We have some differences in the variables included for the different MAP dimensions between the old and new measurement types. Therefore, we create two separate dictionaries that list the respective variables for each MAP dimension and measurement type.
dim_columns_new = {
    'Budgeting_Planning': Budgeting_Planning_columns,
    'Cost': Cost_columns,
    'Financing_Investment': Financing_Investment_columns,
    'Operations': Operations_columns,
    'Performance_Internal_Reporting': Performance_Internal_Reporting_columns,
    'Pricing_Revenue_Management': Pricing_Revenue_Management_columns,
    'Risk_Internal_Control': Risk_Internal_Control_columns,
    'Strategy': Strategy_columns
}

dim_columns_old = {
    'Budget': Budgeting_Planning_columns,
    'Cost': Cost_columns,
    'Investment': Financing_Investment_columns,
    'Operations': Operations_columns,
    'Performance': Performance_Internal_Reporting_columns,
    'Risk': Risk_Internal_Control_columns,
    'Strategy': Strategy_columns
}

Optional: We can also investigate the correlations between the relevant columns to identify potential multicollinearity issues before running the random forest analysis. 
We can do this by calculating the correlation matrix and looking for pairs of variables that have a high correlation (e.g., above 0.7 or below -0.7).

In [ ]:
# First, we can select the relevant MAP dimension for the analysis. 
MAP_dim = 'Performance_Internal_Reporting'

# Now, we can select the relevant columns for the selected MAP dimension based on the measurement approach. If the measurement approach is 'BoW', we use the old dimension names, otherwise we use the new dimension names.
if MAP_dim in dim_columns_new:
    relevant_dim_columns = dim_columns_new[MAP_dim]
else:
    relevant_dim_columns = dim_columns_old[MAP_dim]

# Finally, we can create a list of all relevant columns for the analysis, which includes the MAP dimension columns, the industry columns, and the relevant control variables columns.
cols = [col for col in merged_df_encoded.columns if col.startswith(MAP_dim + '_')] + Industry_columns + relevant_dim_columns 

# Show columns with more than 200 missing values in a decreasing order
missing_values = merged_df_encoded[cols].isnull().sum()

display(missing_values[missing_values >= 200].sort_values(ascending=False))

# Calculate the correlation matrix for the selected columns 
correlation_matrix = merged_df_encoded[cols].corr() # exclude the first 14, 16, or 64 columns

high_correlations = correlation_matrix[(correlation_matrix > 0.6) | (correlation_matrix < -0.6)].stack().reset_index()
high_correlations = high_correlations[high_correlations['level_0'] != high_correlations['level_1']]
high_correlations.columns = ['Variable 1', 'Variable 2', 'Correlation']

high_correlations['Abs Correlation'] = high_correlations['Correlation'].abs()
high_correlations = high_correlations.sort_values(by='Abs Correlation', ascending=False).drop(columns=['Abs Correlation'])

# Delete duplicate pairs
high_correlations = high_correlations.drop_duplicates(subset=['Correlation'])

# Display the high correlations
display(high_correlations)

### Random Forest

Now we can run the random forest analysis using the relevant columns for the selected MAP dimension and the industry dummies as features, and the tf-idf (CS) weighted MAP dimension as the target variable. Bootstrapping is used to estimate the variability of feature importance across different samples of the data, which can help us identify which features are consistently important across different subsets of the data. In total, we will run 50 bootstrap iterations per MAP dimension and then aggregate the results to identify the most important features across the bootstraps. We will look at permutation importance and SHAP importance, which are both model-agnostic methods for assessing feature importance. Permutation importance measures the decrease in model performance when a feature's values are randomly shuffled, while SHAP values provide a unified measure of feature importance based on cooperative game theory. To be more precise, SHAP values quantify the contribution of each feature to the prediction for each individual observation by considering all possible combinations of features. This allows SHAP values to capture both the main effects of individual features and their interactions with other features, providing a more comprehensive understanding of feature importance compared to permutation importance, which only captures the main effects of features.

First, we define a function that runs one single random forest regression for a given dependent variable and a set of independent variables. 

In [ ]:
# Define a function to run a single bootstrap iteration
def run_single_bootstrap(b, X, Y_value):
    X_tr, X_te, Y_tr, Y_te = train_test_split(
        X, Y_value, test_size=0.30, random_state=b
    )
    # Train Random Forest model
    rf = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=10,
        max_features='sqrt',
        n_jobs=1,          # <<< IMPORTANT
        random_state=b
    )
    rf.fit(X_tr, Y_tr)

    # Permutation importance (out-of-sample)
    perm = permutation_importance(
        rf, X_te, Y_te,
        n_repeats=10,
        n_jobs=1           # <<< IMPORTANT
    )

    perm_imp = (
        pd.Series(perm.importances_mean, index=X.columns)
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'index': 'variable', 0: 'importance'})
    )

    # SHAP importance (out-of-sample)
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_te)
    mean_shap = np.abs(shap_values).mean(axis=0)

    shap_imp = (
        pd.Series(mean_shap, index=X.columns)
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'index': 'variable', 0: 'importance'})
    )

    return perm_imp, shap_imp

Now, we can run the random forest analysis with permutation importance and SHAP values for each MAP dimension separately. We will run 50 bootstraps for each MAP dimension and save the results in separate "feature importance" plots, one for each approach respectively.  

In [ ]:
# Define the number of bootstraps
N_BOOTSTRAPS = 50

# Get the number of CPU cores available for parallel processing
n_jobs = multiprocessing.cpu_count()

# Now we loop through the different MAP dimensions and run the random forest analysis with permutation importance and SHAP values for each dimension separately. 

for MAP_dim in MAP_dims: # options: 'Budgeting_Planning', 'Cost', 'Financing_Investment', 'Operations', 'Performance_Internal_Reporting', 'Pricing_Revenue_Management', 'Risk_Internal_Control', 'Strategy'
    print(f'Analyzing MAP dimension: {MAP_dim}')

    if measurement_approach == 'BoW':
        relevant_dim_columns = dim_columns_old[MAP_dim]
    else:
        relevant_dim_columns = dim_columns_new[MAP_dim]

    # We include in the analysis all industry dummies, the respective variables for the MAP dimension under investigation, 
    # and potentially correlated variables that we deem relevant for the analysis
    cols = [col for col in merged_df_encoded.columns if col.startswith(MAP_dim + '_')] + Industry_columns + relevant_dim_columns

    # Next, we remove rows with missing values
    analysis_df = merged_df_encoded[cols].dropna()

    # Save the number of observations used for the analysis
    num_observations = analysis_df.shape[0]
    print(f'Number of observations used for the analysis: {num_observations}')

    # Define feature matrix X and target variable Y
    if measurement_approach == 'GLLM':
        X = analysis_df.iloc[:, 8:] # all columns except the first 8 MAP-related columns
    else:
        X = analysis_df.iloc[:, 2:] # all columns except the first 2 MAP-related columns
    # Winsorize the continuous features (not binary) to limit the influence of outliers (winsorize at 1% and 99%).
    # Note that we winsorize the continuous features once we have already dropped the missing values, so we do not need to worry about missing values in the winsorization process.
    for col in X.columns:
        if not X[col].isin([0, 1]).all():
            # convert the feature variable to an float array and winsorize at the 1st and 99th percentile to reduce the influence of outliers 
            X[col] = winsorize(np.asarray(X[col], dtype=float), limits=[0.01, 0.01])

    # Depending on the measurement approach, we have different MAP dimensions and respective variables that we include in the analysis. 
    if measurement_approach == 'GLLM':
        Y_equally = analysis_df.iloc[:, :4]     # :4 for GLLM --> equally weighted MAP dimensions

        Y_tf_idf = analysis_df.iloc[:, 4:8]  # 4:8 for GLLM --> CS-weighted MAP dimensions
    else:
        Y_equally = analysis_df.iloc[:, :1]     # :1 for Bow and W2V --> equally weighted MAP dimensions

        Y_tf_idf = analysis_df.iloc[:, 1:2]  # 1:2 for BoW and for W2V --> tf-idf weighted MAP dimensions

    # Finally, we run the random forest analysis with permutation importance and SHAP values for each MAP dimension separately. 
    # We run 50 bootstraps for each MAP dimension and save the results in separate dataframes for permutation importance and SHAP values, respectively. 
    for dim in Y_tf_idf.columns:

        print(f'Starting permutation importance via random forest for dimension: {dim}')

        Y_value = analysis_df[dim]

        # Run bootstraps in parallel
        results = Parallel(n_jobs=n_jobs-1)(
            delayed(run_single_bootstrap)(b, X, Y_value)
            for b in range(N_BOOTSTRAPS)
        )

        # Split results into separate lists for permutation importance and SHAP importance
        perm_list = [r[0] for r in results]
        shap_list = [r[1] for r in results]

        # Concatenate results into dataframes
        perm_df = pd.concat(perm_list, ignore_index=True)
        shap_df = pd.concat(shap_list, ignore_index=True)

        # Frequency table and average importance scores for Permutation Importance
        freq_perm_df = (perm_df.groupby('variable').agg(frequency=('importance', 'count'), average_importance=('importance', 'mean')).reset_index().sort_values('average_importance', ascending=False))

        # Frequency table and average importance scores for SHAP
        freq_shap_df = (shap_df.groupby('variable').agg(frequency=('importance', 'count'), average_importance=('importance', 'mean')).reset_index().sort_values('average_importance', ascending=False))

        # Plotting stable features based on permutation importance
        plot_feature = (
        plotnine.ggplot(freq_perm_df, plotnine.aes(x='variable', y='average_importance')) + 
        plotnine.geom_bar(stat='identity', fill='skyblue', width=0.8) + 
        plotnine.coord_flip() + 
        plotnine.theme_minimal() +
        plotnine.labs(x='Feature', y='Average Permutation Importance Score', title=f'Stable Features - Permutation Importance for {dim} (N={num_observations})') + 
        plotnine.scale_y_continuous(limits=(0, np.round(np.max(freq_perm_df['average_importance'])+0.01, 2))) + 
        plotnine.scale_x_discrete(limits=freq_perm_df.sort_values('average_importance', ascending=True)['variable']) + 
        plotnine.theme(
            panel_background=plotnine.element_rect(fill='white'),
            plot_background=plotnine.element_rect(fill='white'),
            axis_title_x=plotnine.element_text(size=18, weight='bold'),
            axis_title_y=plotnine.element_text(size=18, weight='bold'),
            axis_text_x=plotnine.element_text(size=12, weight='bold'),
            axis_text_y=plotnine.element_text(size=12, weight='bold')
            )
        )
        # Save the plot for permutation importance
        if within_industry:
            plot_feature.save(f'{plots_dir}/{measurement_approach}/Stable_features_permutation_importance_{dim}_within_industry_plot.png', width=19.2, height=9.67, dpi=300)
        elif without_finance:
            plot_feature.save(f'{plots_dir}/{measurement_approach}/Stable_features_permutation_importance_{dim}_without_finance_plot.png', width=19.2, height=9.67, dpi=300)
        else:
            plot_feature.save(f'{plots_dir}/{measurement_approach}/Stable_features_permutation_importance_{dim}_plot.png', width=19.2, height=9.67, dpi=300)

        # Plotting stable features based on SHAP importance
        plot_shap = (
        plotnine.ggplot(freq_shap_df, plotnine.aes(x='variable', y='average_importance')) +
        plotnine.geom_bar(stat='identity', fill='salmon', width=0.8) +
        plotnine.coord_flip() + 
        plotnine.theme_minimal() +
        plotnine.labs(x='Feature', y='Average SHAP Importance Score', title=f'Stable Features - SHAP Importance for {dim} (N={num_observations})') + 
        plotnine.scale_y_continuous(limits=(0, np.round(np.max(freq_shap_df['average_importance'])+0.005, 2))) + 
        plotnine.scale_x_discrete(limits=freq_shap_df.sort_values('average_importance', ascending=True)['variable']) + 
        plotnine.theme(
            panel_background=plotnine.element_rect(fill='white'),
            plot_background=plotnine.element_rect(fill='white'),
            axis_title_x=plotnine.element_text(size=18, weight='bold'),
            axis_title_y=plotnine.element_text(size=18, weight='bold'),
            axis_text_x=plotnine.element_text(size=12, weight='bold'),
            axis_text_y=plotnine.element_text(size=12, weight='bold')
            )
        )
        # Save the plot for SHAP importance
        if within_industry:
            plot_shap.save(f'{plots_dir}/{measurement_approach}/Stable_features_SHAP_importance_{dim}_within_industry_plot.png', width=19.2, height=9.67, dpi=300)
        elif without_finance:
            plot_shap.save(f'{plots_dir}/{measurement_approach}/Stable_features_SHAP_importance_{dim}_without_finance_plot.png', width=19.2, height=9.67, dpi=300)
        else:
            plot_shap.save(f'{plots_dir}/{measurement_approach}/Stable_features_SHAP_importance_{dim}_plot.png', width=19.2, height=9.67, dpi=300)

del merged_df_encoded, analysis_df, X, Y_equally, Y_tf_idf, perm_df, shap_df, freq_perm_df, freq_shap_df, plots_dir, plot_feature, plot_shap

### OLS Regressions

Now, OLS regressions are conducted in which the dependent variables are the candidate variables from above and the key independent variable is the MAP dimension score corresponding to the respective measurement method. In addition, we control for firm size, return on totla assets, as well as time and industry fixed effects.

First, we need to prepare the data for the regression analyses. Firm "Size" is measured as the natural logarithm of total assets and return on assets ("ROA") is already in the dataset.

In [ ]:
# Size is measured as log of total assets, which is a common measure of firm size in the literature and is also used as a control variable in the regression analyses for the convergent validity checks
merged_df['Size'] = np.log(winsorize(merged_df['Total Assets'], limits=[0.01, 0.01], nan_policy='omit'))

# ROA is measured as net income divided by total assets, which is a common measure of firm performance in the literature and is also used as a control variable in the regression analyses for the convergent validity checks
# --> already in dataset as 'Return on Average Total Assets - %, TTM'
# We rename this variable to 'ROA' for easier use in the regression analyses
merged_df = merged_df.rename(columns={'Return on Average Total Assets - %, TTM': 'ROA'})

# Winsorize ROA at the 1st and 99th percentile to reduce the influence of outliers
merged_df['ROA'] = pd.to_numeric(merged_df['ROA'], errors='coerce')/100  
merged_df['ROA'] = winsorize(merged_df['ROA'], limits=[0.01, 0.01], nan_policy='omit')

# As year we take the 'Period End Year' variable, which indicates the year of the financial statement that we used for the analysis. 
# As industry we take the 'NAICS Sector Name' variable, which indicates the industry of the firm based on the NAICS classification.
# For these variables, we do not need to winsorize them and use them as they are in the regression analyses.


# MAP cols are the first 2:16 (2:66) columns of the merged dataframe, which contain the MAP dimensions (both equally weighted and tf-idf weighted) that we use as independent variables in the regression analyses for the convergent validity checks
if measurement_approach == 'BoW':
    MAP_cols = merged_df.columns[2:16].tolist() 
elif measurement_approach.startswith('W2V'):
    MAP_cols = merged_df.columns[2:18].tolist() 
else:
    MAP_cols = merged_df.columns[2:66].tolist() 

# Winsorize the MAP columns at the 1st and 99th percentile to reduce the influence of outliers
for col in MAP_cols:
    merged_df[col] = winsorize(merged_df[col], limits=[0.01, 0.01], nan_policy='omit')

# Define a function to run the regression analyses for the convergent validity checks. 

def run_validation_regression(df, outcome, map_var, time_col='Period_End_Year', industry_col='NAICS_Sector_Name',):
    # Define the regression formula
    formula = f'''
        {outcome} ~ {map_var}
        + Size
        + ROA
        + C({time_col})
        + C({industry_col})
    '''
    # Create a copy of the dataframe to avoid modifying the original dataframe
    df = df.copy()

    # Remove rows with missing values in the relevant columns for the regression analysis
    required_cols = [
        outcome,
        map_var,
        'Size',
        'ROA',
        time_col,
        industry_col
    ]
    df = df[required_cols].dropna()

    # Fit the regression model with clustered standard errors at the industry level
    model = smf.ols(formula, data=df).fit(
        cov_type='cluster',
        cov_kwds={'groups': df[industry_col]}  
    )

    return model


Second, we create a summary statistic for all dependent "candidat" variables,

In [ ]:
summary_df = merged_df.copy()

# Create a summary statistics for all dependent variables of the dimensions: Budgeting_Planning to Strategy (without the industry dummies) and save it as an excel file
summary_df = summary_df[Budgeting_Planning_columns + Cost_columns + Financing_Investment_columns + Operations_columns + Performance_Internal_Reporting_columns + Pricing_Revenue_Management_columns + Risk_Internal_Control_columns + Strategy_columns + ['pat_grant_fyear', 'pat_app_fyear']].copy()

# Create empty lists to store the summary statistics for each variable
summary_stats = []

# Loop through each variable in the summary_df and calculate the summary statistics
for col in summary_df.columns:

    # Drop missing values for the current column
    col_values = summary_df[col].dropna()

    # If multiple columns are selected, drop all columns except one to avoid issues with the describe() function
    if isinstance(col_values, pd.DataFrame) and col_values.shape[1] > 1:
        col_values = col_values.iloc[:, 0]

    #Winsorize all variables in the summary_df at the 1st and 99th percentile
    if not col_values.isin([0, 1]).all():
        # convert the feature variable to an float array and winsorize at the 1st and 99th percentile to reduce the influence of outliers 
        col_values = pd.Series(winsorize(np.asarray(col_values, dtype=float), limits=[0.01, 0.01], nan_policy='omit'))

    # Calculate the summary statistics for the current column
    summary_stat = col_values.describe().to_dict()

    # Add the variable name to the summary statistics dictionary
    summary_stat['variable'] = col
    summary_stats.append(summary_stat)

# Create a summary statistics table for the summary_df and make sure that the variable name is the first column (remove duplicates if there are any) and save it as an excel file
summary_stats_df = pd.DataFrame(summary_stats)
summary_stats_df = summary_stats_df[['variable'] + [col for col in summary_stats_df.columns if col != 'variable']]
summary_df = summary_stats_df.drop_duplicates(subset=['variable']).reset_index(drop=True)
display(summary_stats_df)

# Just save it once when looking at the corpus of "W2V_v1" (results will be the same for all corpora, since the same merged dataframe is used for all corpora)
if measurement_approach == 'BoW':
    summary_stats_df.to_excel(f'{tables_dir}/Summary_Statistics_Concurrent_Convergent_Validity.xlsx', index=False)

del summary_df, summary_stat, summary_stats, col, col_values, summary_stats_df

Last, we run the regression analyses.

In [ ]:
# Run OLS regression for each combination of outcome variable and MAP variable

# Adjust the strategy columns to include the log of the number of patent grants and applications.
Strategy_columns = ['MnA_Deal_Indicator', 'Num_MnA_Deals', 'Net Cash Flow from Investing Activities', 
'Acquisition & Disposal of Business Sold/(Acquired) Net - CF', 'EPS - Diluted - excl Exord Items Applicable to Common Total',
'Management Training', 'Integrated Strategy in MD&A Score', 'Training and Development Policy', 'Reinvestment Rate - %, TTM', 'Management Score', 
'log_grant_pat', 'log_app_pat', 'R&D Ratio'] 

# We have some differences in the variables included for the different MAP dimensions between the old and new measurement types. Therefore, we create two separate dictionaries that list the respective variables for each MAP dimension and measurement type.
dim_columns_new = {
    'Budgeting_Planning': Budgeting_Planning_columns,
    'Cost': Cost_columns,
    'Financing_Investment': Financing_Investment_columns,
    'Operations': Operations_columns,
    'Performance_Internal_Reporting': Performance_Internal_Reporting_columns,
    'Pricing_Revenue_Management': Pricing_Revenue_Management_columns,
    'Risk_Internal_Control': Risk_Internal_Control_columns,
    'Strategy': Strategy_columns
}

dim_columns_old = {
    'Budget': Budgeting_Planning_columns,
    'Cost': Cost_columns,
    'Investment': Financing_Investment_columns,
    'Operations': Operations_columns,
    'Performance': Performance_Internal_Reporting_columns,
    'Risk': Risk_Internal_Control_columns,
    'Strategy': Strategy_columns
}

# create directory for the results if it does not exist
if not os.path.exists(f'{tables_dir}/{measurement_approach}'):
    os.makedirs(f'{tables_dir}/{measurement_approach}')

# loop through the MAP variables and outcome variables to run the regression analyses for the convergent validity checks

if measurement_approach == 'BoW':
    tf_idf_cols = MAP_cols[7:]
elif measurement_approach.startswith('W2V'):
    tf_idf_cols = MAP_cols[8:]
else:
    tf_idf_cols = MAP_cols[32:] 

for map_var in tf_idf_cols:

    results_list = []

    if measurement_approach == 'BoW':
        MAP_dim = map_var.split('_tf_idf_standardized')[0]
        outcome_vars = dim_columns_old[MAP_dim]
    elif measurement_approach.startswith('W2V'):
        MAP_dim = map_var.split('_tf_idf_standardized')[0]
        outcome_vars = dim_columns_new[MAP_dim]
    else:
        if map_var.endswith('_explicit_CS_standardized') or map_var.endswith('_explicit_CS_standardized_FT'):
            MAP_dim = map_var.split('_explicit_CS_standardized')[0]
        elif map_var.endswith('_implicit_CS_standardized') or map_var.endswith('_implicit_CS_standardized_FT'):
            MAP_dim = map_var.split('_implicit_CS_standardized')[0]
        outcome_vars = dim_columns_new[MAP_dim]
    
    for outcome_var in outcome_vars: 

        analysis_df = merged_df[[outcome_var , 'Instrument', 'Period End Year', 'NAICS Sector Name', 'Size', 'ROA'] + MAP_cols].dropna().reset_index(drop=True)

        #drop duplicate columns
        analysis_df = analysis_df.loc[:,~analysis_df.columns.duplicated()]

        # delete rows with missing values in the relevant columns for the regression analysis
        analysis_df = analysis_df.dropna(subset=[outcome_var, 'Size', 'ROA', 'Period End Year', 'NAICS Sector Name', map_var]) 

        # calculate the standard deviation of the outcome variable and check if it is an indicator variable (0,1 ; or 0.0,1.0) or a continuous variable
        outcome_std = analysis_df[outcome_var].std()

        indicator_check = analysis_df[outcome_var].isin([0, 1]).all()

        # if the outcome is not an indicator (0,1 ; or 0.0,1.0), winsorize the outcome variable at the 1st and 99th percentile to reduce the influence of outliers
        standardizer = StandardScaler()
        # convert the outcome variable to an float array and winsorize at the 1st and 99th percentile to reduce the influence of outliers 
        if not indicator_check:
            analysis_df[outcome_var] = winsorize(np.asarray(analysis_df[outcome_var], dtype=float), limits=[0.01, 0.01])
        # we also standardize the outcome variable to make the coefficients of the regression analyses for the convergent validity checks more comparable across different outcome variables (since they are measured on different scales)
        analysis_df[outcome_var] = standardizer.fit_transform(analysis_df[[outcome_var]])
            
        # rename columns so that they do not have spaces in their names, which can cause issues with the regression formula
        analysis_df = analysis_df.rename(columns={outcome_var: outcome_var.replace('%', '').replace(' ', '_').replace('&', 'and').replace('/', '_').replace('(', '').replace(')', '').replace('-', '_').replace(',', '_').replace('+', '_'), # remove special characters from the outcome variable name for easier use in the regression formula
                                                'Period End Year': 'Period_End_Year', 
                                                'NAICS Sector Name': 'NAICS_Sector_Name'})     
        # If the number of observation is less than 1000, we skip the regression analysis
        num_observations = analysis_df.shape[0]
        if num_observations < 800:
            print(f'Skipping regression for outcome variable {outcome_var} and MAP variable {map_var} due to insufficient observations ({num_observations} observations).')
            continue

        try:
            result = run_validation_regression(analysis_df, outcome_var.replace('%', '').replace(' ', '_').replace('&', 'and').replace('/', '_').replace('(', '').replace(')', '').replace('-', '_').replace(',', '_').replace('+', '_'), map_var)

        except Exception as e:
            print(f'Error running regression for outcome variable {outcome_var}: {e}')
            continue

        # save the coefficients, standard errors, t-values, and p-values of the map variable to a dataframe
        coef_df = {
            'map_variable': map_var,
            'outcome': outcome_var,
            'coefficient': result.params[map_var],
            'std_error': result.bse[map_var],
            't_value': result.tvalues[map_var],
            'p_value': result.pvalues[map_var],
            'sample_std': outcome_std,
            'indicator_check': indicator_check,
            'summary': result.summary().as_text(),
            'num_observations': num_observations
        }

        results_list.append(coef_df)

    results_df = pd.DataFrame(results_list)

    #sort results_df by p-value in ascending order
    results_df = results_df.sort_values('p_value', ascending=True)

    results_df.to_excel(f'{tables_dir}/{measurement_approach}/Concurrent_Convergent_validity_results_{map_var}.xlsx', index=False)

del tf_idf_cols, results_list, results_df, outcome_vars, outcome_var, analysis_df, outcome_std, indicator_check, standardizer, num_observations, result, coef_df

## Discriminant Validity

To assess the discriminant validity of the new measures, pairwise correlations of the single MAP dimensions are investigated. The rationale behind this is that the individual MAP dimensions capture clearly different constructs while still belonging to one overarching theoretical concept. Therefore, the dimensional measures should show some, but not perfect, correlations, indicating that they reflect distinct facets of an MA system.

Load the merged corpus with the normalized MAP measures. We can choose between the different measurement approaches and normalization procedures by changing the 'measurement_approach', 'wihtin_industry', and 'without_finance' variables below. 

NOTE: In the paper just the baseline configuration (within_industry = False, without_finance = True) is shown. But the same can be done with the other possible settings.

In [ ]:
# Load the merged corpus with the normalized MAP measures and the additional variables
# We can choose between the different measurement types by changing the 'measurement_approach', 'within_industry' and 'without_finance' variables below.

# Specifiy the measurement approach to use for the analysis. Options are: 'BoW', 'W2V_v1', 'W2V_v2', 'GLLM'
measurement_approach = 'GLLM'

# Specify whether to use MAP measures normalized across the whole sample (False) or within each industry (True)
within_industry = False  

# Specify whether to exclude firms operating in the Financing/Investment sector (true or false).
without_finance = True

# Load the merged dataframe for the respective measurement type
if within_industry:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_within_industry_normalized.pkl')
elif without_finance:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_without_finance_normalized.pkl')
else:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_final.pkl')

# drop duplicates in column 'Merge Key' if there are any (we just want one observation per firm-year for the upcoming analyses)
if merged_df['Merge Key'].duplicated().sum() > 0:
    merged_df = merged_df.drop_duplicates(subset=['Merge Key'], keep='last').reset_index(drop=True)

Next we remove columns that are not relevant for the discriminant validity analysis. Basically everything except the MAP dimension columns. In addition, we define a function that helps us to calculate the p-values for the correlation tables in the next step.

In [ ]:
# TF-IDF (CS) MAP cols 
if measurement_approach == 'GLLM':
    MAP_cols = [col for col in merged_df.columns.tolist() if col.endswith('_explicit_CS_standardized') or col.endswith('_implicit_CS_standardized') or col.endswith('_explicit_CS_standardized_FT') or col.endswith('_implicit_CS_standardized_FT')]

else:
    MAP_cols = [col for col in merged_df.columns.tolist() if col.endswith('_tf_idf_standardized')]

# Filter next for the tf-idf weighted MAP columns that we want to use for the discriminant validity checks. We only include the tf-idf weighted MAP columns that are relevant for the respective measurement type.
merged_df = merged_df[MAP_cols].copy()

# Winsorize the tf-idf weighted MAP columns at the 1st and 99th percentile to reduce the influence of outliers
for col in MAP_cols:
    merged_df[col] = winsorize(merged_df[col], limits=[0.01, 0.01], nan_policy='omit')

# Function to calculate p-values
def calculate_pvalues(correlation_matrix, original_df):
    df = correlation_matrix.dropna()._get_numeric_data()
    pvalues = pd.DataFrame(np.ones((df.shape[1], df.shape[1])), columns=df.columns, index=df.columns)
    for row in df.columns:
        for col in df.columns:
            if row != col:
                pvalues.loc[row, col] = pearsonr(original_df[row], original_df[col])[1]

    return pvalues


Last, we perform the discriminant vailidty checks by calculating a Pearson correlation matrix among the MAP dimensions.

In [ ]:
# Next we create a correlation matrix for each measurement type across the different MAP dimensions to check if the they measure different constructs (i.e., different MAP dimensions) and save the correlation matrix as a heatmap figure.
# we select the columns that start with the measurement type and drop rows with missing values in these columns for the correlation analysis

# Mapping for renaming the columns and index
MAP_mapping = {'Strategy': 'Strategy', 'Budgeting_Planning': 'Budgeting/Planning', 'Cost': 'Cost', 'Operations': 'Operations', 
                'Financing_Investment': 'Financing/Investment', 'Performance_Internal_Reporting': 'Performance/Internal Reporting', 
                'Pricing_Revenue_Management': 'Pricing/Revenue Management', 'Risk_Internal_Control': 'Risk/Internal Control'}

if measurement_approach == 'GLLM':
    # calculate the correlation matrix for the 4 different types of GLLM measures '_explicit_CS_standardized', '_explicit_CS_standardized_FT', '_implicit_CS_standardized', '_implicit_CS_standardized_FT' separately to check if the different different types of measures (explicit vs. implicit) are correlated with each other, which would indicate that they are measuring the same underlying construct (i.e., the MAP dimension).
    GLLM_measure_types = ['explicit_CS_standardized', 'explicit_CS_standardized_FT', 'implicit_CS_standardized', 'implicit_CS_standardized_FT']
    for GLLM_measure_type in GLLM_measure_types:

        merged_df_tmp = merged_df[[col for col in merged_df.columns if col.endswith(f'_{GLLM_measure_type}')]].copy()

        # change variable names by cropping off '_explicit_CS_standardized', '_explicit_CS_standardized_FT', '_implicit_CS_standardized', or '_implicit_CS_standardized_FT'
        merged_df_tmp.columns = [col.replace('_explicit_CS_standardized_FT', '').replace('_explicit_CS_standardized', '').replace('_implicit_CS_standardized_FT', '').replace('_implicit_CS_standardized', '') for col in merged_df_tmp.columns]
            
        # Order columns in the same order as the MAP dimensions to make the heatmap easier to read
        ordered_columns = ['Budgeting_Planning', 'Cost', 'Financing_Investment', 'Operations', 'Performance_Internal_Reporting', 'Pricing_Revenue_Management', 'Risk_Internal_Control', 'Strategy']
        merged_df_tmp = merged_df_tmp[[col for col in ordered_columns if col in merged_df_tmp.columns]]

        # Calculate the correlation matrix
        correlation_matrix = merged_df_tmp.corr(method='pearson')

        # Calculate the p-values
        pvalues = calculate_pvalues(correlation_matrix, merged_df_tmp)
        # Create a mask for the upper triangle
        mask = np.tril(np.ones_like(correlation_matrix, dtype=bool))

        correlation_matrix.columns = [MAP_mapping.get(col) for col in correlation_matrix.columns]
        correlation_matrix.index = [MAP_mapping.get(idx) for idx in correlation_matrix.index]

        # Set up the matplotlib figure
        plt.figure(figsize=(12, 6))
        # Draw the heatmap with the mask and correct aspect ratio
        sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, cbar_kws={'shrink': .8})
        # Annotate significance levels
        for i in range(len(correlation_matrix.columns)):
            for j in range(i+1, len(correlation_matrix.columns)):
                p = pvalues.iloc[i, j]
                if p < 0.01:
                    plt.text(j+0.7, i+0.5, '***', ha='left', va='center', color='black', fontsize=12)
                elif p < 0.05:
                    plt.text(j+0.7, i+0.5, '**', ha='left', va='center', color='black', fontsize=12)
                elif p < 0.1:
                    plt.text(j+0.7, i+0.5, '*', ha='left', va='center', color='black', fontsize=12)
        # Make x and y axis ticks bold
        plt.xticks(fontsize=9, fontweight='bold', rotation=45, ha='right')
        plt.yticks(fontsize=9, fontweight='bold')
        # Save the heatmap
        if within_industry:
            plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_{measurement_approach}_{GLLM_measure_type}_within_industry.png', dpi=300, bbox_inches='tight')
        elif without_finance:
            plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_{measurement_approach}_{GLLM_measure_type}_without_finance.png', dpi=300, bbox_inches='tight')
        else:
            plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_{measurement_approach}_{GLLM_measure_type}.png', dpi=300, bbox_inches='tight')
    
else:

    # change variable names by cropping off '_tf_idf_standardized' from the column names to make the heatmap easier to read
    merged_df.columns = [col.replace(f'_tf_idf_standardized', '') for col in merged_df.columns]

    # Order columns in the same order as the MAP dimensions to make the heatmap easier to read
    if measurement_approach == 'BoW':
        ordered_columns = ['Budget', 'Cost', 'Investment', 'Operations', 'Performance', 'Risk', 'Strategy']
    else:
        ordered_columns = ['Budgeting_Planning', 'Cost', 'Financing_Investment', 'Operations', 'Performance_Internal_Reporting', 'Pricing_Revenue_Management', 'Risk_Internal_Control', 'Strategy']
    merged_df = merged_df[[col for col in ordered_columns if col in merged_df.columns]]
        
    # Calculate the correlation matrix
    correlation_matrix = merged_df.corr(method='pearson')

    # Calculate the p-values
    pvalues = calculate_pvalues(correlation_matrix, merged_df)
    # Create a mask for the upper triangle
    mask = np.tril(np.ones_like(correlation_matrix, dtype=bool))

    # Rename the columns and index if not Bow
    if measurement_approach != 'BoW':
        correlation_matrix.columns = [MAP_mapping.get(col) for col in correlation_matrix.columns]
        correlation_matrix.index = [MAP_mapping.get(idx) for idx in correlation_matrix.index]

    # Set up the matplotlib figure
    plt.figure(figsize=(12, 6))
    # Draw the heatmap with the mask and correct aspect ratio
    sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, cbar_kws={'shrink': .8})

    # Annotate significance levels 
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            p = pvalues.iloc[i, j]
            if p < 0.01:
                plt.text(j+0.7, i+0.5, '***', ha='left', va='center', color='black', fontsize=12)
            elif p < 0.05:
                plt.text(j+0.7, i+0.5, '**', ha='left', va='center', color='black', fontsize=12)
            elif p < 0.1:
                plt.text(j+0.7, i+0.5, '*', ha='left', va='center', color='black', fontsize=12)
        
    # Make x and y axis ticks bold
    plt.xticks(fontsize=9, fontweight='bold', rotation=45, ha='right')
    plt.yticks(fontsize=9, fontweight='bold')

    # Save the heatmap
    if within_industry:
        plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_MAP_dimensions_{measurement_approach}_within_industry.png', dpi=300, bbox_inches='tight')
    elif without_finance:
        plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_MAP_dimensions_{measurement_approach}_without_finance.png', dpi=300, bbox_inches='tight')
    else:
        plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_MAP_dimensions_{measurement_approach}.png', dpi=300, bbox_inches='tight')

## Predictive Validity Analysis 

The final validity assessment is the evaluation of predictive validity. Specifically, the informational and signaling value of MAP disclosure is examined by estimating OLS regressions of firm performance measures on MAP-fit scores, while controlling for a broad set of covariates.

Load the merged corpus with the normalized MAP measures. We can choose between the different measurement approaches and normalization procedures by changing the 'measurement_approach', 'wihtin_industry', and 'without_finance' variables below. 

NOTE: In the paper different configurations are reported. The main results are for the configurations (within_industry = False, without_finance = True) and (within_industry = True, without_finance = False) using 'measurement_method = 'tf_idf_standardized''.

In [ ]:
# Load the merged corpus with the normalized MAP measures and the additional variables
# We can choose between the different measurement types by changing the 'measurement_approach', 'measurement_method', 'within_industry' and 'without_finance' variables below.

# Specifiy the measurement approach to use for the analysis. Options are: 'BoW', 'W2V_v1', 'W2V_v2', 'GLLM'
measurement_approach = 'GLLM'

# Specify the measurement method to use for the analysis. Options are: 'tf_idf_standardized' and 'equally_weighted_standardized'
measurement_method = 'tf_idf_standardized'

# Specify whether to use MAP measures normalized across the whole sample (False) or within each industry (True)
within_industry = False  

# Specify whether to exclude firms operating in the Financing/Investment sector (true or false).
without_finance = True

# Load the merged dataframe for the respective measurement type
if within_industry:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_within_industry_normalized.pkl')
elif without_finance:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_without_finance_normalized.pkl')
else:
    merged_df = pd.read_pickle(f'{measurement_approach.split("_")[0]}/Validation_Merge_{measurement_approach}_final.pkl')

# drop duplicates in column 'Merge Key' if there are any (we just want one observation per firm-year for the upcoming analyses)
if merged_df['Merge Key'].duplicated().sum() > 0:
    merged_df = merged_df.drop_duplicates(subset=['Merge Key'], keep='last').reset_index(drop=True)

Before, we can turn to the final regression analyses, we need to create the MAP-fit measure.

In [ ]:
# Now, we create the MAP-fit measure by calculating the cosine similarity between the disclosed MAP scores and 
# the theoretically optimal MAP scores for each MAP dimension, which are based on life cycle stages.

# To do so, we first need to define the theoretically optimal MAP score vectors in the following order:
# Using the Bow Method: [Budget, Cost, Investment, Operations, Performance, Risk, Strategy]

optimal_MAP_vectors_BoW = {
    'Birth': [0, 0, 0, 0, 0, 0, 1],
    'Growth': [1, 0.5, 1, 1, 1, 1, 0.5],
    'Mature': [0.5, 1, 0.5, 1, 0.5, 0.5, 0.5],
    'Decline': [0.5, 0, 1, 0, 0.5, 1, 1],
    'Revive': [0, 1, 0, 0, 0, 0, 0]
}

# Using the W2V and GLLM Method: [Budgeting_Planning, Cost, Financing_Investment, Operations, Performance_Internal_Reporting, Pricing_Revenue_Management, Risk_Internal_Control, Strategy]

optimal_MAP_vectors_W2V_GLLM = {
    'Birth': [0, 0, 0, 0, 0, 0, 0, 1],
    'Growth': [1, 0.5, 1, 1, 1, 0.5, 1, 0.5],
    'Mature': [0.5, 1, 0.5, 1, 0.5, 1, 0.5, 0.5],
    'Decline': [0.5, 0, 1, 0, 0.5, 0, 1, 1],
    'Revive': [0, 1, 0, 0, 0, 1, 0, 0]
}

# Next, we calculate the cosine similarity between the disclosed MAP score vector and the optimal MAP score vector 

# delet rows with missing values in 'Life Cycle Stage 1' and 'Life Cycle Stage 2' columns
merged_df = merged_df.dropna(subset=['Life Cycle Stage 1', 'Life Cycle Stage 2'])


if measurement_approach == 'BoW':
    MAP_dims = ['Budget', 'Cost', 'Investment', 'Operations', 'Performance', 'Risk', 'Strategy']
    if measurement_method == 'tf_idf_standardized':
        MAP_cols = [f'{dim}_tf_idf_standardized' for dim in MAP_dims]
    else:
        MAP_cols = [f'{dim}_equally_standardized' for dim in MAP_dims]

    #drop rows with missing values in the MAP columns
    merged_df = merged_df.dropna(subset=MAP_cols)

    # now create a list of tuples (optimal MAP vector, disclosed MAP vector) 
    MAP_fit_vectors_1 = [(optimal_MAP_vectors_BoW[row['Life Cycle Stage 1']], row[MAP_cols].values.tolist()) for _, row in merged_df.iterrows()]
    MAP_fit_vectors_2 = [(optimal_MAP_vectors_BoW[row['Life Cycle Stage 2']], row[MAP_cols].values.tolist()) for _, row in merged_df.iterrows()]
    # calculate cosine similarity for each tuple and create a new column in the merged dataframe for the MAP-fit measure for Life Cycle Stage 1 and Life Cycle Stage 2
    merged_df['MAP_fit_1'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_1]
    merged_df['MAP_fit_2'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_2]
    
elif measurement_approach.startswith('W2V'):
    MAP_dims = ['Budgeting_Planning', 'Cost', 'Financing_Investment', 'Operations', 'Performance_Internal_Reporting', 'Pricing_Revenue_Management', 'Risk_Internal_Control', 'Strategy']
    if measurement_method == 'tf_idf_standardized':
        MAP_cols = [f'{dim}_tf_idf_standardized' for dim in MAP_dims]
    else:
        MAP_cols = [f'{dim}_equally_standardized' for dim in MAP_dims]

    #drop rows with missing values in the MAP columns
    merged_df = merged_df.dropna(subset=MAP_cols)

    # now create a list of tuples (optimal MAP vector, disclosed MAP vector)
    MAP_fit_vectors_1 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 1']], row[MAP_cols].values.tolist()) for _, row in merged_df.iterrows()]
    MAP_fit_vectors_2 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 2']], row[MAP_cols].values.tolist()) for _, row in merged_df.iterrows()]
    merged_df['MAP_fit_1'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_1]
    merged_df['MAP_fit_2'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_2]

else:
    MAP_dims = ['Budgeting_Planning', 'Cost', 'Financing_Investment', 'Operations', 'Performance_Internal_Reporting', 'Pricing_Revenue_Management', 'Risk_Internal_Control', 'Strategy']

    if measurement_method == 'tf_idf_standardized':
        MAP_cols_explicit_CS = [f'{dim}_explicit_CS_standardized' for dim in MAP_dims]
        MAP_cols_explicit_CS_FT = [f'{dim}_explicit_CS_standardized_FT' for dim in MAP_dims]
        MAP_cols_implicit_CS = [f'{dim}_implicit_CS_standardized' for dim in MAP_dims]
        MAP_cols_implicit_CS_FT = [f'{dim}_implicit_CS_standardized_FT' for dim in MAP_dims]
    else:
        MAP_cols_explicit_CS = [f'{dim}_explicit_equally_standardized' for dim in MAP_dims]
        MAP_cols_explicit_CS_FT = [f'{dim}_explicit_equally_standardized_FT' for dim in MAP_dims]
        MAP_cols_implicit_CS = [f'{dim}_implicit_equally_standardized' for dim in MAP_dims]
        MAP_cols_implicit_CS_FT = [f'{dim}_implicit_equally_standardized_FT' for dim in MAP_dims]

    #drop rows with missing values in the MAP columns
    merged_df = merged_df.dropna(subset=MAP_cols_explicit_CS + MAP_cols_explicit_CS_FT + MAP_cols_implicit_CS + MAP_cols_implicit_CS_FT)
    
    # now create a list of tuples (optimal MAP vector, disclosed MAP vector) for explicit CS normalized variables
    MAP_fit_vectors_1 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 1']], row[MAP_cols_explicit_CS].values.tolist()) for _, row in merged_df.iterrows()]
    MAP_fit_vectors_2 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 2']], row[MAP_cols_explicit_CS].values.tolist()) for _, row in merged_df.iterrows()]
    merged_df['MAP_fit_1_explicit'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_1]
    merged_df['MAP_fit_2_explicit'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_2]

    # now create a list of tuples (optimal MAP vector, disclosed MAP vector) for explicit CS FT variables
    MAP_fit_vectors_1 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 1']], row[MAP_cols_explicit_CS_FT].values.tolist()) for _, row in merged_df.iterrows()]
    MAP_fit_vectors_2 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 2']], row[MAP_cols_explicit_CS_FT].values.tolist()) for _, row in merged_df.iterrows()]
    merged_df['MAP_fit_1_explicit_FT'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_1]
    merged_df['MAP_fit_2_explicit_FT'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_2]

    # now create a list of tuples (optimal MAP vector, disclosed MAP vector) for implicit CS normalized variables
    MAP_fit_vectors_1 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 1']], row[MAP_cols_implicit_CS].values.tolist()) for _, row in merged_df.iterrows()]
    MAP_fit_vectors_2 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 2']], row[MAP_cols_implicit_CS].values.tolist()) for _, row in merged_df.iterrows()]
    merged_df['MAP_fit_1_implicit'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_1]
    merged_df['MAP_fit_2_implicit'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_2]

    # now create a list of tuples (optimal MAP vector, disclosed MAP vector) for implicit CS FT variables
    MAP_fit_vectors_1 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 1']], row[MAP_cols_implicit_CS_FT].values.tolist()) for _, row in merged_df.iterrows()]
    MAP_fit_vectors_2 = [(optimal_MAP_vectors_W2V_GLLM[row['Life Cycle Stage 2']], row[MAP_cols_implicit_CS_FT].values.tolist()) for _, row in merged_df.iterrows()]
    merged_df['MAP_fit_1_implicit_FT'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_1]
    merged_df['MAP_fit_2_implicit_FT'] = [cosine_similarity([vec1], [vec2])[0][0] for vec1, vec2 in MAP_fit_vectors_2]

del optimal_MAP_vectors_BoW, optimal_MAP_vectors_W2V_GLLM, MAP_fit_vectors_1, MAP_fit_vectors_2

if measurement_approach == 'GLLM':
    del MAP_cols_explicit_CS, MAP_cols_explicit_CS_FT, MAP_cols_implicit_CS, MAP_cols_implicit_CS_FT
else:
    del MAP_cols

Next, we need to prepare the final sample for the predictive analyses. 

First, we create the analyses dataset by dropping missings in the needed main and control variables.

In [ ]:
# create the analysis dataframe for the regression analyses for the predictive validity checks by keeping only the relevant columns for the regression analyses and dropping rows with missing values in the relevant columns

if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
     relevant_columns = ['MAP_fit_1', 'MAP_fit_2', 'Return on Average Total Assets - %, TTM', 'Return on Average Total Equity - %', 'Total Assets', 
                         'Gross Profit Margin - %', 'Operating Margin - %', 'EPS - Diluted - excl Exord Items Applicable to Common Total',
                         'Total Debt Percentage of Total Assets', 'Revenue Growth', 'Stock Return Volatility', 'Age', 'Largest Shareholder', 
                         'CEO Chairman Duality', 'filing_year', 'Period End Year', 'Period End Month', 'NAICS Sector Name', 'Instrument', 'Total Return',
                         'ROA_lag_t+1', 'EPS_lag_t+1', 'Gross_Margin_lag_t+1', 'Operating_Margin_lag_t+1',
                         'ROA_lag_t+2', 'EPS_lag_t+2', 'Gross_Margin_lag_t+2', 'Operating_Margin_lag_t+2',
                         'ROA_lag_t+3', 'EPS_lag_t+3', 'Gross_Margin_lag_t+3', 'Operating_Margin_lag_t+3',
                         'ROA_lag_t+4', 'EPS_lag_t+4', 'Gross_Margin_lag_t+4', 'Operating_Margin_lag_t+4',
                         'ROA_lag_t+5', 'EPS_lag_t+5', 'Gross_Margin_lag_t+5', 'Operating_Margin_lag_t+5',
                         'ROA_lag_t+6', 'EPS_lag_t+6', 'Gross_Margin_lag_t+6', 'Operating_Margin_lag_t+6',
                         'ROA_lag_t_minus_1', 'EPS_lag_t_minus_1', 'Gross_Margin_lag_t_minus_1', 'Operating_Margin_lag_t_minus_1']
else:
     relevant_columns = ['MAP_fit_1_explicit', 'MAP_fit_2_explicit', 'MAP_fit_1_explicit_FT', 'MAP_fit_2_explicit_FT', 'MAP_fit_1_implicit', 'MAP_fit_2_implicit', 'MAP_fit_1_implicit_FT', 'MAP_fit_2_implicit_FT',
                   'Return on Average Total Assets - %, TTM', 'Return on Average Total Equity - %', 'Total Assets', 'Total Debt Percentage of Total Assets', 
                   'Gross Profit Margin - %', 'Operating Margin - %', 'EPS - Diluted - excl Exord Items Applicable to Common Total', 'Revenue Growth', 
                   'Stock Return Volatility', 'Age', 'Largest Shareholder', 'CEO Chairman Duality', 'filing_year', 'Period End Year', 'Period End Month', 'NAICS Sector Name', 'Instrument', 'Total Return',
                   'ROA_lag_t+1', 'EPS_lag_t+1', 'Gross_Margin_lag_t+1', 'Operating_Margin_lag_t+1',
                   'ROA_lag_t+2', 'EPS_lag_t+2', 'Gross_Margin_lag_t+2', 'Operating_Margin_lag_t+2',
                    'ROA_lag_t+3', 'EPS_lag_t+3', 'Gross_Margin_lag_t+3', 'Operating_Margin_lag_t+3',
                    'ROA_lag_t+4', 'EPS_lag_t+4', 'Gross_Margin_lag_t+4', 'Operating_Margin_lag_t+4',
                    'ROA_lag_t+5', 'EPS_lag_t+5', 'Gross_Margin_lag_t+5', 'Operating_Margin_lag_t+5',
                    'ROA_lag_t+6', 'EPS_lag_t+6', 'Gross_Margin_lag_t+6', 'Operating_Margin_lag_t+6',
                   'ROA_lag_t_minus_1', 'EPS_lag_t_minus_1', 'Gross_Margin_lag_t_minus_1', 'Operating_Margin_lag_t_minus_1']

analysis_df = merged_df[relevant_columns].copy()

# Drop missing values in the most relevant columns (main outcome variable, main MAP variable, and key controls) to create the analysis dataframe for the regression analyses for the predictive validity checks
CONTROLS = ['Total Assets', 'Total Debt Percentage of Total Assets', 'Revenue Growth', 'Age', 'Total Return', 'Largest Shareholder', 'CEO Chairman Duality', 'Stock Return Volatility']
outcome_variables = ['Return on Average Total Assets - %, TTM', 'EPS - Diluted - excl Exord Items Applicable to Common Total', 'Operating Margin - %']

map_variables = [col for col in analysis_df.columns if col.startswith('MAP_fit_1')]

analysis_df = analysis_df.copy().dropna(subset=outcome_variables + map_variables + CONTROLS)

# lastly, we also drop single observations, since we are using firm fixed effects in the regression analyses for the predictive validity checks, which require at least two observations per firm. We drop single observations based on the 'Instrument' column, which contains the unique identifier for each firm.
analysis_df = analysis_df.groupby('Instrument').filter(lambda x: len(x) > 1).reset_index(drop=True)

del merged_df, outcome_variables, map_variables

Second, we define the control variables, transform percentage values to dicimal numbers, and winsorize all continous variables

In [ ]:

# Size is measured as log of total assets, which is a common measure of firm size in the literature and is also used as a control variable in the regression analyses for the convergent validity checks
analysis_df['Size'] = np.log(analysis_df['Total Assets'])

# remove the original Total Assets variable from the analysis dataframe
analysis_df = analysis_df.drop(columns=['Total Assets'])

# Age is measured as natural log of 1 plus the number of years since the firm's founding (or IPO if founding year is not available)
analysis_df['Age_log'] = np.log(1 + analysis_df['Age'])
# remove the original Age variable from the analysis dataframe
analysis_df = analysis_df.drop(columns=['Age'])

# rename columns to make them easier to work with in the regression analyses for the convergent validity checks
analysis_df = analysis_df.rename(columns={'Period End Year': 'Period_End_Year', 
                                          'Period End Month': 'Period_End_Month',
                                          'NAICS Sector Name': 'NAICS_Sector_Name',
                                          'NAICS Subsector Name': 'NAICS_Subsector_Name',
                                          'Return on Average Total Assets - %, TTM': 'ROA',
                                          'Return on Average Total Equity - %': 'ROE',
                                          'EPS - Diluted - excl Exord Items Applicable to Common Total': 'EPS',
                                          'Gross Profit Margin - %': 'Gross_Margin',
                                          'Operating Margin - %': 'Operating_Margin',
                                          'Total Debt Percentage of Total Assets': 'Leverage',
                                          'Revenue Growth': 'Growth',
                                          'Total Return': 'Return',
                                          'Largest Shareholder': 'Top1shareholder',
                                          'CEO Chairman Duality': 'Duality',
                                          'Stock Return Volatility': 'Volatility'
                                          })   

# simplyfy the renmaing of the lagged variables by replacing 'lag_t_+_' with 'lag_t_plus_' and 'lag_t_-_1' with 'lag_t_minus_1'
analysis_df.columns = analysis_df.columns.str.replace('lag_t+', 'lag_t_plus_')
analysis_df.columns = analysis_df.columns.str.replace('lag_t-', 'lag_t_minus_')

# change percentage variables to numeric and divide by 100 to convert to proportions
analysis_df['ROA'] = pd.to_numeric(analysis_df['ROA'], errors='coerce') / 100
analysis_df['ROA_lag_t_minus_1'] = pd.to_numeric(analysis_df['ROA_lag_t_minus_1'], errors='coerce') / 100
analysis_df['ROE'] = pd.to_numeric(analysis_df['ROE'], errors='coerce') / 100
analysis_df['Gross_Margin'] = pd.to_numeric(analysis_df['Gross_Margin'], errors='coerce') / 100
analysis_df['Gross_Margin_lag_t_minus_1'] = pd.to_numeric(analysis_df['Gross_Margin_lag_t_minus_1'], errors='coerce') / 100
analysis_df['Operating_Margin'] = pd.to_numeric(analysis_df['Operating_Margin'], errors='coerce') / 100
analysis_df['Operating_Margin_lag_t_minus_1'] = pd.to_numeric(analysis_df['Operating_Margin_lag_t_minus_1'], errors='coerce') / 100
analysis_df['EPS'] = pd.to_numeric(analysis_df['EPS'], errors='coerce') 

for i in range(1, 7):
    analysis_df[f'ROA_lag_t_plus_{i}'] = pd.to_numeric(analysis_df[f'ROA_lag_t_plus_{i}'], errors='coerce') / 100
    analysis_df[f'Gross_Margin_lag_t_plus_{i}'] = pd.to_numeric(analysis_df[f'Gross_Margin_lag_t_plus_{i}'], errors='coerce') / 100
    analysis_df[f'Operating_Margin_lag_t_plus_{i}'] = pd.to_numeric(analysis_df[f'Operating_Margin_lag_t_plus_{i}'], errors='coerce') / 100
    analysis_df[f'EPS_lag_t_plus_{i}'] = pd.to_numeric(analysis_df[f'EPS_lag_t_plus_{i}'], errors='coerce')

analysis_df['Leverage'] = pd.to_numeric(analysis_df['Leverage'], errors='coerce') / 100
analysis_df['Growth'] = pd.to_numeric(analysis_df['Growth'], errors='coerce') / 100
analysis_df['Return'] = pd.to_numeric(analysis_df['Return'], errors='coerce') / 100
analysis_df['Volatility'] = pd.to_numeric(analysis_df['Volatility'], errors='coerce') / 100

# winsorize all continuous variables at the 1st and 99th percentile to reduce the influence of outliers
# NOTE: THE LAGGED VARIABLES WILL BE WINSORIZED AT A LATER STAGE (BEFORE THE REGRESSION ANALYSES)

if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    map_vars = ['MAP_fit_1', 'MAP_fit_2'] # MAP_fit_1 and MAP_fit_2
else:
    map_vars = ['MAP_fit_1_explicit', 'MAP_fit_2_explicit', 'MAP_fit_1_explicit_FT', 'MAP_fit_2_explicit_FT', 'MAP_fit_1_implicit', 'MAP_fit_2_implicit', 'MAP_fit_1_implicit_FT', 'MAP_fit_2_implicit_FT']

for map_var in map_vars:
    analysis_df[map_var] = winsorize(analysis_df[map_var], limits=[0.01, 0.01], nan_policy='omit')

analysis_df['ROE'] = winsorize(analysis_df['ROE'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['ROA'] = winsorize(analysis_df['ROA'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['ROA_lag_t_minus_1'] = winsorize(analysis_df['ROA_lag_t_minus_1'], limits=[0.01, 0.01], nan_policy='omit')

analysis_df['Gross_Margin'] = winsorize(analysis_df['Gross_Margin'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['Gross_Margin_lag_t_minus_1'] = winsorize(analysis_df['Gross_Margin_lag_t_minus_1'], limits=[0.01, 0.01], nan_policy='omit')

analysis_df['Operating_Margin'] = winsorize(analysis_df['Operating_Margin'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['Operating_Margin_lag_t_minus_1'] = winsorize(analysis_df['Operating_Margin_lag_t_minus_1'], limits=[0.01, 0.01], nan_policy='omit')

analysis_df['EPS'] = winsorize(analysis_df['EPS'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['EPS_lag_t_minus_1'] = winsorize(analysis_df['EPS_lag_t_minus_1'], limits=[0.01, 0.01], nan_policy='omit')

analysis_df['Size'] = winsorize(analysis_df['Size'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['Age_log'] = winsorize(analysis_df['Age_log'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['Leverage'] = winsorize(analysis_df['Leverage'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['Growth'] = winsorize(analysis_df['Growth'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['Return'] = winsorize(analysis_df['Return'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['Top1shareholder'] = winsorize(analysis_df['Top1shareholder'], limits=[0.01, 0.01], nan_policy='omit')
analysis_df['Volatility'] = winsorize(analysis_df['Volatility'], limits=[0.01, 0.01], nan_policy='omit')

del map_var, map_vars

Third, we have a look at summary statistics for the relevant variables and have a look at the mean MAP-fit scores across filing years and NAICS industry sectors.

In [ ]:
# create a summary statistics for all the relevant variables for the regression analyses for the predictive validity checks

if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    summary_df = analysis_df[['MAP_fit_1', 'ROA', 'EPS', 'Operating_Margin', 'Size', 'Leverage', 'Growth', 'Age_log', 'Return', 'Top1shareholder', 'Duality', 'Volatility']].copy().dropna()
else:
    summary_df = analysis_df[['MAP_fit_1_explicit', 'MAP_fit_1_explicit_FT', 'MAP_fit_1_implicit', 'MAP_fit_1_implicit_FT', 'ROA', 'EPS', 'Operating_Margin', 'Size', 'Leverage', 'Growth', 'Age_log', 'Return', 'Top1shareholder', 'Duality', 'Volatility']].copy().dropna()

# rounds the summary statistics to 3 decimal places
summary_stats = summary_df.describe().transpose().round(3)

display(summary_stats)

# Save the summary statistics to an excel file
if without_finance:
    summary_stats.to_excel(f'{tables_dir}/{measurement_approach}/Summary_Statistics_Predictive_Validity_{measurement_method}_without_finance.xlsx')
elif within_industry:
    summary_stats.to_excel(f'{tables_dir}/{measurement_approach}/Summary_Statistics_Predictive_Validity_{measurement_method}_within_industry.xlsx')
else:
    summary_stats.to_excel(f'{tables_dir}/{measurement_approach}/Summary_Statistics_Predictive_Validity_{measurement_method}.xlsx')

# Create summary of the categorical variables 'NAICS_Sector_Name' and 'filing_year' by displaying the count of observations for each category in these variables and then calculate the mean MAP_fit_1 for each category in these variables and display the results in a table
print('Summary of NAICS_Sector_Name:')

if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    summary_df = analysis_df[['MAP_fit_1', 'NAICS_Sector_Name', 'filing_year', 'ROA', 'EPS', 'Operating_Margin', 'Size', 'Leverage', 'Growth', 'Age_log', 'Return', 'Top1shareholder', 'Duality', 'Volatility']].copy().dropna()
else:
    summary_df = analysis_df[['MAP_fit_1_implicit_FT', 'NAICS_Sector_Name', 'filing_year', 'ROA', 'EPS', 'Operating_Margin', 'Size', 'Leverage', 'Growth', 'Age_log', 'Return', 'Top1shareholder', 'Duality', 'Volatility']].copy().dropna()

# Create a summary of the categorical variable 'NAICS_Sector_Name' by displaying the count of observations for each category in this variable
industry_summary = summary_df['NAICS_Sector_Name'].value_counts()

#Add mean and standard deviation of MAP_fit_1 for each category in 'NAICS_Sector_Name' to the industry_summary dataframe
if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    industry_summary = pd.DataFrame(industry_summary)
    industry_summary['Mean_MAP_fit_1'] = summary_df.groupby('NAICS_Sector_Name')['MAP_fit_1'].mean().round(3)
    industry_summary['Std_MAP_fit_1'] = summary_df.groupby('NAICS_Sector_Name')['MAP_fit_1'].std().round(3)
else:
    industry_summary = pd.DataFrame(industry_summary)
    industry_summary['Mean_MAP_fit_1_implicit_FT'] = summary_df.groupby('NAICS_Sector_Name')['MAP_fit_1_implicit_FT'].mean().round(3)
    industry_summary['Std_MAP_fit_1_implicit_FT'] = summary_df.groupby('NAICS_Sector_Name')['MAP_fit_1_implicit_FT'].std().round(3)

# sort the industry_summary dataframe by names of the NAICS_Sector_Name in alphabetical order
industry_summary = industry_summary.sort_index()

# Save the industry_summary to an excel file
if without_finance:
    industry_summary.to_excel(f'{tables_dir}/{measurement_approach}/Summary_MAP_fit_{measurement_method}_by_NAICS_Sector_Name_without_finance.xlsx')
elif within_industry:
    industry_summary.to_excel(f'{tables_dir}/{measurement_approach}/Summary_MAP_fit_{measurement_method}_by_NAICS_Sector_Name_within_industry.xlsx')
else:
    industry_summary.to_excel(f'{tables_dir}/{measurement_approach}/Summary_MAP_fit_{measurement_method}_by_NAICS_Sector_Name.xlsx')

display(industry_summary)
# Create a summary of the categorical variable 'filing_year' by displaying the count of observations for each category in this variable
year_summary = summary_df['filing_year'].value_counts()

if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    year_summary = pd.DataFrame(year_summary)
    year_summary['Mean_MAP_fit_1'] = summary_df.groupby('filing_year')['MAP_fit_1'].mean().round(3)
    year_summary['Std_MAP_fit_1'] = summary_df.groupby('filing_year')['MAP_fit_1'].std().round(3)
else:
    year_summary = pd.DataFrame(year_summary)
    year_summary['Mean_MAP_fit_1_implicit_FT'] = summary_df.groupby('filing_year')['MAP_fit_1_implicit_FT'].mean().round(3)
    year_summary['Std_MAP_fit_1_implicit_FT'] = summary_df.groupby('filing_year')['MAP_fit_1_implicit_FT'].std().round(3)

# sort the year_summary dataframe by names of the filing_year in alphabetical order
year_summary = year_summary.sort_index()

# Save the year_summary to an excel file
if without_finance:
    year_summary.to_excel(f'{tables_dir}/{measurement_approach}/Summary_MAP_fit_{measurement_method}_by_filing_year_without_finance.xlsx')
elif within_industry:
    year_summary.to_excel(f'{tables_dir}/{measurement_approach}/Summary_MAP_fit_{measurement_method}_by_filing_year_within_industry.xlsx')
else:
    year_summary.to_excel(f'{tables_dir}/{measurement_approach}/Summary_MAP_fit_{measurement_method}_by_filing_year.xlsx')

display(year_summary)

del summary_stats, summary_df, industry_summary, year_summary

Fourth, we perform a correlation analysis across all conisdered variables and a variance inflaction factor analysis. This is mainly done to detect potential multicollinearity among control variables and correlations among MAP-fit scores and firm performance.

In [ ]:
# Check correlation among the relevant control plus main performance columns (without 'NAICS Sector Name', 'Instrument') 
# for the regression analyses for the predictive validity checks

CONTROLS = [
    'Size', 'Leverage', 'Growth', 'Age_log',
    'Return', 'Top1shareholder', 'Duality', 'Volatility'
]
if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    performance_metrics = [
    'MAP_fit_1', 'MAP_fit_2','ROA', 'EPS', 'Operating_Margin'
    ]
else:
    performance_metrics = [
     'MAP_fit_1_implicit_FT', 'MAP_fit_1_explicit', 'ROA',  'EPS', 'Operating_Margin' #, 'MAP_fit_1_explicit', 'MAP_fit_2_explicit', 'MAP_fit_1_explicit_FT', 'MAP_fit_2_explicit_FT', 'MAP_fit_1_implicit', 'MAP_fit_2_implicit',  'MAP_fit_2_implicit_FT',
    ]

correlation_df = analysis_df[performance_metrics + CONTROLS].copy().dropna()

# winsorize the correlation dataframe at the 1st and 99th percentile to reduce the influence of outliers (without MAP-fit variables, which are already winsorized)
for col in correlation_df.columns:
    if col.startswith('MAP_fit'):
        continue
    else:
        correlation_df[col] = winsorize(correlation_df[col], limits=[0.01, 0.01], nan_policy='omit')

correlation_matrix = correlation_df.corr(method='pearson')

# Function to calculate p-values
def calculate_pvalues(df):
    df = df.dropna()._get_numeric_data()
    pvalues = pd.DataFrame(np.ones((df.shape[1], df.shape[1])), columns=df.columns, index=df.columns)
    for row in df.columns:
        for col in df.columns:
            if row != col:
                pvalues.loc[row, col] = pearsonr(correlation_df[row], correlation_df[col])[1]
    return pvalues


# Calculate p-values
pvalues = calculate_pvalues(correlation_matrix)

# Create a mask for the upper triangle
mask = np.tril(np.ones_like(correlation_matrix, dtype=bool))

# display the correlation matrix as a heatmap

# Set up the matplotlib figure
plt.figure(figsize=(10, 8))
# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)

# Annotate significance
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        p = pvalues.iloc[i, j]
        if p < 0.01:
            plt.text(j+0.7, i+0.5, '***', ha='left', va='center', color='black', fontsize=12)
        elif p < 0.05:
            plt.text(j+0.7, i+0.5, '**', ha='left', va='center', color='black', fontsize=12)
        elif p < 0.1:
            plt.text(j+0.7, i+0.5, '*', ha='left', va='center', color='black', fontsize=12)


# Make x and y axis ticks bold
plt.xticks(fontsize=9, fontweight='bold', rotation=45, ha='right')
plt.yticks(fontsize=9, fontweight='bold')

plt.title('Correlation Matrix of Relevant Variables for Predictive Validity Checks')

# Save the correlation matrix heatmap to a file
if without_finance:
    plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_Predictive_Validity_{measurement_method}_without_finance.png', bbox_inches='tight', dpi=300)
elif within_industry:
    plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_Predictive_Validity_{measurement_method}_within_industry.png', bbox_inches='tight', dpi=300)
else:
    plt.savefig(f'{plots_dir}/{measurement_approach}/Correlation_Matrix_Predictive_Validity_{measurement_method}.png', bbox_inches='tight', dpi=300)


plt.show()

del correlation_df, correlation_matrix, pvalues, mask, calculate_pvalues, i, j, p

In [ ]:
# Check the Variance Inflation Factor (VIF) for the relevant control plus main performance columns 

# 1. Prepare the design matrix (X) 
# This must match exactly what you put into the OLS model
# If you used a formula like 'ROA ~ MAP_fit_1 + Size + ...', use dmatrices
from patsy import dmatrices

VIF_df = analysis_df[performance_metrics + CONTROLS + ['Period_End_Year', 'NAICS_Sector_Name', 'Instrument']].copy().dropna()

# Create the design matrices for the VIF calculation, remove the intercept term from the design matrix (X) since it is not needed for VIF calculation
if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    y, X = dmatrices('ROA ~ 0 + C(Period_End_Year) + C(NAICS_Sector_Name) + MAP_fit_1 + '
                 'Size + Leverage + Growth + Age_log + Return + '
                 'Top1shareholder + Duality + Volatility',
                 data=VIF_df, return_type='dataframe')
    
else:
    y, X = dmatrices('ROA ~ 0 + C(Period_End_Year) + C(NAICS_Sector_Name) + MAP_fit_1_implicit_FT + '
                 'Size + Leverage + Growth + Age_log + Return + '
                 'Top1shareholder + Duality + Volatility', 
                 data=VIF_df, return_type='dataframe')

# 2. Calculate VIF for each variable, except the fixed effects (Period_End_Year and NAICS_Sector_Name) since they are not relevant for VIF calculation
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

display(vif_data.sort_values('VIF', ascending=False))

del VIF_df, y, X, vif_data

Next, we will check whether there is enough variation in the MAP_fit measure, in order to include firm fixed effects at a later stage.

In [ ]:
# Check the panel structure and within-variation diagnostics for the relevant MAP variables
if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    map_vars = relevant_columns[:2] # MAP_fit_1 and MAP_fit_2
else:
    map_vars = relevant_columns[:8] # MAP_fit_1_explicit, MAP_fit_2_explicit, MAP_fit_1_explicit_FT, MAP_fit_2_explicit_FT, MAP_fit_1_implicit, MAP_fit_2_implicit, MAP_fit_1_implicit_FT, MAP_fit_2_implicit_FT

for map_var in map_vars:

    # Set the panel index
    panel_df = analysis_df.set_index(['Instrument', 'Period_End_Year']).sort_index()

    # ── Panel structure ───────────────────────────────────────────────────────────
    total_obs     = panel_df[map_var].notna().sum()
    n_firms       = panel_df.index.get_level_values('Instrument').nunique()
    n_years       = panel_df.index.get_level_values('Period_End_Year').nunique()
    obs_per_firm  = panel_df.groupby(level='Instrument')[map_var].count()

    print('=' * 55)
    print('PANEL STRUCTURE')
    print('=' * 55)
    print(f'  Firms (N):              {n_firms}')
    print(f'  Years (T):              {n_years}')
    print(f'  Total observations:     {total_obs}')
    print(f'  Avg obs per firm:       {obs_per_firm.mean():.2f}')
    print(f'  Median obs per firm:    {obs_per_firm.median():.1f}')
    print(f'  Min obs per firm:       {obs_per_firm.min()}')
    print(f'  Max obs per firm:       {obs_per_firm.max()}')
    print(f'  Firms with ≥2 obs:      {(obs_per_firm >= 2).sum()}  '
        f'({(obs_per_firm >= 2).mean()*100:.1f}% — usable for FE)')
    print(f'  Balanced panel:         '
        f'{'Yes' if (obs_per_firm == n_years).all() else 'No'}')

    # ── Variation decomposition (Stata-style) ─────────────────────────────────────
    # Overall variance
    overall_var = panel_df[map_var].var(ddof=1)

    # Between variance: variance of firm means (std dev of firm means)
    firm_means   = panel_df.groupby(level='Instrument')[map_var].mean()
    between_var  = firm_means.var(ddof=1)   # variance across firm means

    # Within variance: average of per-firm variances (demeaned)
    #   Equivalent to: variance of (x_it - x_i_bar + x_bar)
    grand_mean   = panel_df[map_var].mean()
    demeaned     = (panel_df[map_var]
                    - panel_df.groupby(level='Instrument')[map_var]
                            .transform('mean')
                    + grand_mean)
    within_var   = demeaned.var(ddof=1)

    print('\n' + '=' * 55)
    print(f"VARIANCE DECOMPOSITION  (Stata xtsum convention) for '{map_var}'")
    print('=' * 55)
    print(f'{'Type':<12} {'Variance':>12} {'Std Dev':>12} {'% of Overall':>14}')
    print('-' * 55)
    for label, var in [('Overall', overall_var),
                    ('Between', between_var),
                    ('Within',  within_var)]:
        sd  = np.sqrt(var)
        pct = var / overall_var * 100
        print(f'  {label:<10} {var:>12.4f} {sd:>12.4f} {pct:>13.1f}%')

    # ── Within-variation diagnostics ─────────────────────────────────────────────
    within_ratio = within_var / overall_var

    print('\n' + '=' * 55)
    print(f"WITHIN-VARIATION DIAGNOSTICS for '{map_var}'")
    print('=' * 55)
    print(f'  Within / Overall ratio: {within_ratio:.3f}  '
        f'({'✓ good' if within_ratio > 0.1 else '⚠ low — FE may struggle'})')

    # Firms that never change (zero within-variance)
    firm_std = panel_df.groupby(level='Instrument')[map_var].std()
    n_no_change = (firm_std == 0).sum()
    n_low_change = (firm_std < firm_std.quantile(0.1)).sum()

    print(f'  Firms with zero within-var: {n_no_change} '
        f'({n_no_change/n_firms*100:.1f}%)')
    print(f'  Firms in bottom 10% of within-var: {n_low_change}')
    print(f'  Mean within-firm std dev:   {firm_std.mean():.4f}')
    print(f'  Median within-firm std dev: {firm_std.median():.4f}')

    # Distribution of within-firm std devs
    print(f'\n  Within-firm std dev distribution:')
    for q, label in [(0.10,'p10'), (0.25,'p25'), (0.50,'p50'),
                    (0.75,'p75'), (0.90,'p90')]:
        print(f'    {label}: {firm_std.quantile(q):.4f}')

    print('\n' + '=' * 55)
    if within_ratio > 0.2:
        print('✓ Sufficient within variation — firm FE should be reliable.')
    elif within_ratio > 0.1:
        print('⚠ Moderate within variation — FE feasible but check SE inflation.')
    else:
        print('✗ Low within variation — FE will absorb most signal; reconsider.')
    print('=' * 55)

    del panel_df, overall_var, between_var, within_var, within_ratio, firm_means, demeaned, firm_std, grand_mean, n_no_change
    del n_low_change, obs_per_firm, total_obs, n_firms, n_years, pct, q, label, var, sd
else:
    pass

Now, we define some helpe functions to run the different regression analyses.

In [ ]:
# Define a shared helper function to prepare the dataframe for regression: copy, log-transform outcome if needed, select relevant columns, and dropna on those columns only (not the entire dataframe)
def _prepare_df(df, outcome_col, map_var, controls, extra_cols):
    '''Shared helper: copy, log-transform, dropna on relevant columns only.'''
    df = df.copy()

    keep = list(dict.fromkeys([outcome_col, map_var] + controls + extra_cols)) 
    return df[keep].dropna()
# The main regression function with flexible FE specifications and clustered SEs by firm.
def run_validation_regression(
    df,
    outcome,
    map_var,
    controls=CONTROLS,
    time_col='Period_End_Year',
    industry_col='NAICS_Sector_Name',
    firm_col='Instrument',
    firm_FE=True,
    industry_FE=False
):
    '''
    OLS with clustered SEs (by firm).
    - firm_FE=False and industry_FE=False : year fixed effects only (shown in output)
    - firm_FE=False and industry_FE=True : industry + year fixed effects  (shown in output)
    - firm_FE=True and industry_FE=False : firm + year fixed effects, firm FE absorbed & hidden
    - firm_FE=True and industry_FE=True : firm + industry + year fixed effects, if no collinearity issues (firm FE still absorbed & hidden, industry FE shown in output)
    '''
    controls_str = ' + '.join(controls)

    extra_cols = [time_col, firm_col, industry_col]
    df = _prepare_df(df, outcome, map_var, controls, extra_cols)

    # Winsorize the outcome variable at 1% and 99% to mitigate outliers
    df[outcome] = winsorize(df[outcome], limits=[0.01, 0.01], nan_policy='omit')

    # winsorize the control variables at 1% and 99% to mitigate outliers 
    for control in controls:
        df[control] = winsorize(df[control], limits=[0.01, 0.01], nan_policy='omit')

    if firm_FE == True and industry_FE == False:
        # pyfixest absorbs firm and year FE — neither appears in coefficient table
        formula = (
            f'{outcome} ~ {map_var} + {controls_str}'
            f' | {firm_col} + {time_col}'        # FE specification
        )
        model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})

    elif industry_FE == True and firm_FE == False:
        # Industry + year FE via dummies (visible in output, but usually fine)
        formula = (
            f'{outcome} ~ {map_var} + {controls_str}'
            f' | {industry_col} + {time_col}'
        )
        model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})

    elif firm_FE == True and industry_FE == True:
        # Try to include industry FE as dummies, but if too many categories (collinearity), drop industry FE and warn
        formula = (
            f'{outcome} ~ {map_var} + {controls_str}'
            f' | {firm_col} + {industry_col} + {time_col}'
        )
        try:
            model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})
        except Exception as e:
            print(f'Warning: Could not include industry FE due to error: {e}. Running regression without industry FE.')
            formula = (
                f'{outcome} ~ {map_var} + {controls_str}'
                f' | {firm_col} + {time_col}'
            )
            model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})
    else:
        # Year FE only (visible in output)
        formula = (
            f'{outcome} ~ {map_var} + {controls_str}'
            f' | {time_col}'
        )
        model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})

    return model
# The lagged regression function with flexible FE specifications and clustered SEs by firm, including the lagged performance variable as an additional control to test whether MAP predicts future performance even after controlling for past performance.
def run_validation_regression_lagged(
    df,
    outcome,
    outcome_lag,
    map_var,
    time_col='Period_End_Year',
    industry_col='NAICS_Sector_Name',
    firm_col='Instrument'
):
    '''
    Lagged regression: outcome_t ~ outcome_t-1 + MAP + controls + industry/year FE.
    '''
    df = df.copy()  # copy BEFORE any renaming

    controls_str = ' + '.join(CONTROLS)

    formula = (
        f'{outcome} ~ {outcome_lag}'
        f' + {map_var}'
        f' + {controls_str}'
        f' | {industry_col} + {time_col}'
    )

    extra_cols = [time_col, firm_col, industry_col, outcome_lag]
    df = _prepare_df(df, outcome, map_var, CONTROLS, extra_cols)

    # winsorize the outcome variable at 1% and 99% to mitigate outliers (after log-transform if applicable)
    df[outcome] = winsorize(df[outcome], limits=[0.01, 0.01], nan_policy='omit')

    # winsorize the control variables at 1% and 99% to mitigate outliers (after log-transform if applicable)
    for control in CONTROLS:
        df[control] = winsorize(df[control], limits=[0.01, 0.01], nan_policy='omit')

    # remove singeltons (firms with only one observation) to make it comparable to regressions with firm fixed effects (which automatically drop singletons)
    firm_counts = df[firm_col].value_counts()
    df = df[df[firm_col].isin(firm_counts[firm_counts > 1].index)]

    model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})
    return model
# The quartile regression function with flexbile FE specifications and clustered SEs by firm, splitting the MAP variable into quartiles to test for non-linear relationships between MAP and the outcome variable.
def run_validation_regression_quartiles(
    df,
    outcome,
    map_var,
    time_col='Period_End_Year',
    industry_col='NAICS_Sector_Name',
    firm_col='Instrument',
    firm_FE=False,
    quartiles=4
):
    '''
    Quartile regression: outcome_t ~ MAP_quartiles + controls + industry/year FE.
    MAP variable is split into quartiles to test for non-linear relationships.
    '''
    df = df.copy()  # copy BEFORE any renaming
    # Prepare the dataframe: log-transform outcome if needed, select relevant columns, and dropna on those columns only (not the entire dataframe)
    extra_cols = [time_col, firm_col, industry_col]
    df = _prepare_df(df, outcome, map_var, CONTROLS, extra_cols)

    # Winsorize the outcome variable at 1% and 99% to mitigate outliers (after log-transform if applicable)
    df[outcome] = winsorize(df[outcome], limits=[0.01, 0.01], nan_policy='omit')

    # winsorize the control variables at 1% and 99% to mitigate outliers (after log-transform if applicable)
    for control in CONTROLS:
        df[control] = winsorize(df[control], limits=[0.01, 0.01], nan_policy='omit')

    # Create quartile dummies for the MAP variable
    df[f'{map_var}_quartile'] = pd.qcut(df[map_var], q=quartiles, labels=False) + 1  # +1 to make quartiles 1-indexed
    map_quartile_dummies = pd.get_dummies(df[f'{map_var}_quartile'], prefix=f'{map_var}_Q', drop_first=True)  # drop_first to avoid multicollinearity
    df = pd.concat([df, map_quartile_dummies], axis=1)

    # save the number of observations, range, and mean of the MAP variable in each quartile to check for sufficient variation
    obs_per_quartile = df[f'{map_var}_quartile'].value_counts().sort_index()
    mean_per_quartile = df.groupby(f'{map_var}_quartile')[map_var].mean()
    range_per_quartile = df.groupby(f'{map_var}_quartile')[map_var].agg(['min', 'max'])

    # Construct the formula for the regression, including the quartile dummies and controls, and specifying the fixed effects based on the firm_FE parameter
    controls_str = ' + '.join(CONTROLS)
    if firm_FE:
        formula = (
            f'{outcome} ~ {' + '.join(map_quartile_dummies.columns)}'
            f' + {controls_str}'
            f' | {firm_col} + {time_col}'
        )
    else:
        formula = (
            f'{outcome} ~ {' + '.join(map_quartile_dummies.columns)}'
            f' + {controls_str}'
            f' | {industry_col} + {time_col}'
        )
    # Run the regression with clustered standard errors by firm
    model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})
    return model, obs_per_quartile, mean_per_quartile, range_per_quartile
# The persistence regression functions with flexible FE specifications and clustered SEs by firm.
def run_validation_regression_persistence(
    df,
    outcome,
    map_var,
    time_col='Period_End_Year',
    industry_col='NAICS_Sector_Name',
    firm_col='Instrument',
    firm_FE=False
):
    '''
    Persistence regression: outcome_t+h ~ MAP + controls + industry/year FE.
    Outcome column name may contain '+' (e.g. 'ROA_lag_t+1') — handled safely.
    '''
    df = df.copy()  # copy BEFORE any renaming

    # Sanitise column name for formula parser
    if '+' in outcome:
        outcome_col = outcome.replace('+', '_')
        df = df.rename(columns={outcome: outcome_col})
    else:
        outcome_col = outcome

    controls_str = ' + '.join(CONTROLS)
    if firm_FE:
        formula = (
            f'{outcome_col} ~ {map_var}'
            f' + {controls_str}'
            f' | {firm_col} + {time_col}'
        )
    else:
        formula = (
            f'{outcome_col} ~ {map_var}'
            f' + {controls_str}'
            f' | {industry_col} + {time_col}'
        )

    extra_cols = [time_col, firm_col, industry_col]
    df = _prepare_df(df, outcome_col, map_var, CONTROLS, extra_cols)

    # Winsorize the outcome variable at 1% and 99% to mitigate outliers (after log-transform if applicable)
    df[outcome_col] = winsorize(df[outcome_col], limits=[0.01, 0.01], nan_policy='omit')

    # winsorize the control variables at 1% and 99% to mitigate outliers (after log-transform if applicable)
    for control in CONTROLS:
        df[control] = winsorize(df[control], limits=[0.01, 0.01], nan_policy='omit')

    model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})
    return model
# The persistence regression function with flexible FE specifications, clustered SEs by firm, including MAP quartiles.
def run_validation_regression_persistence_quartiles(
    df,
    outcome,
    map_var,
    time_col='Period_End_Year',
    industry_col='NAICS_Sector_Name',
    firm_col='Instrument',
    firm_FE=False,
    quartiles=5
):
    '''
    Persistence regression: outcome_t+h ~ MAP_quartiles + controls + industry/year FE.
    Outcome column name may contain '+' (e.g. 'ROA_lag_t+1') — handled safely.
    '''
    df = df.copy()  # copy BEFORE any renaming

    # Sanitise column name for formula parser
    if '+' in outcome:
        outcome_col = outcome.replace('+', '_')
        df = df.rename(columns={outcome: outcome_col})
    else:
        outcome_col = outcome

    
    # Prepare the dataframe: log-transform outcome if needed, select relevant columns, and dropna on those columns only (not the entire dataframe)
    extra_cols = [time_col, firm_col, industry_col]
    df = _prepare_df(df, outcome_col, map_var, CONTROLS, extra_cols)

    # Winsorize the outcome variable at 1% and 99% to mitigate outliers (after log-transform if applicable)
    df[outcome_col] = winsorize(df[outcome_col], limits=[0.01, 0.01], nan_policy='omit')

    # winsorize the control variables at 1% and 99% to mitigate outliers (after log-transform if applicable)
    for control in CONTROLS:
        df[control] = winsorize(df[control], limits=[0.01, 0.01], nan_policy='omit')

    # Create quartiles for the MAP variable
    df[f'{map_var}_quartile'] = pd.qcut(df[map_var], q=quartiles, labels=False, duplicates='drop') + 1  # +1 to make quartiles 1-indexed
    map_quartile_dummies = pd.get_dummies(df[f'{map_var}_quartile'], prefix=f'{map_var}_Q', drop_first=True)  # drop_first to avoid multicollinearity
    df = pd.concat([df, map_quartile_dummies], axis=1)

    # save the number of observations, range, and mean of the MAP variable in each quartile to check for sufficient variation
    obs_per_quartile = df[f'{map_var}_quartile'].value_counts().sort_index()
    mean_per_quartile = df.groupby(f'{map_var}_quartile')[map_var].mean()
    range_per_quartile = df.groupby(f'{map_var}_quartile')[map_var].agg(['min', 'max'])

    # Construct the formula for the regression, including the quartile dummies and controls, and specifying the fixed effects based on the firm_FE parameter
    controls_str = ' + '.join(CONTROLS)

    if firm_FE:
        formula = (
            f'{outcome_col} ~ {' + '.join(map_quartile_dummies.columns)}'
            f' + {controls_str}'
            f' | {firm_col} + {time_col}'
        )
    else:
        formula = (
            f'{outcome_col} ~ {' + '.join(map_quartile_dummies.columns)}'
            f' + {controls_str}'
            f' | {industry_col} + {time_col}'
        )

    model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})
    return model
# The persistence regression function with flexible FE specifications and clustered SEs by firm, including the lagged performance variable split into quintiles and interacted with MAP to test for heterogeneity in the predictive validity of MAP across different levels of past performance, while avoiding linearity assumptions.
def run_validation_regression_persistence_heterogeneity(
    df,
    outcome,
    map_var,
    time_col='Period_End_Year',
    industry_col='NAICS_Sector_Name',
    firm_col='Instrument',
    firm_FE=False,
    quintiles=5
):
    '''
    Persistence regression: outcome_t+h ~ MAP * perf_t + controls + industry/year FE.
    Outcome column name may contain '+' (e.g. 'ROA_lag_t+1') — handled safely.
    '''
    df = df.copy()  # copy BEFORE any renaming

    # Sanitise column name for formula parser
    if '+' in outcome:
        performance_metric = outcome.split('_lag_')[0]
        outcome_col = outcome.replace('+', '_')
        df = df.rename(columns={outcome: outcome_col})
    else:
        outcome_col = outcome
        performance_metric = outcome.split('_lag_')[0] if '_lag_' in outcome else outcome

    # Create quintile dummies for the performance_metric variable
    df[f'{performance_metric}_quintile'] = pd.qcut(df[performance_metric], q=quintiles, labels=False) + 1  # +1 to make quintiles 1-indexed
    outcome_quintile_dummies = pd.get_dummies(df[f'{performance_metric}_quintile'], prefix=f'{performance_metric}_Q', drop_first=True)  # drop_first to avoid multicollinearity
    df = pd.concat([df, outcome_quintile_dummies], axis=1)

    controls_str = ' + '.join(CONTROLS)
    if firm_FE:
        formula = (
            f'{outcome_col} ~ {map_var}'
            f' + C({f'{performance_metric}_quintile'})'
            f' + C({f'{performance_metric}_quintile'}):{map_var}'   # interaction (no main effects re-added)
            f' + {controls_str}'
            f' | {firm_col} + {time_col}'
        )
    else:
        formula = (
            f'{outcome_col} ~ {map_var}'
            f' + C({f'{performance_metric}_quintile'})'
            f' + C({f'{performance_metric}_quintile'}):{map_var}'   # interaction (no main effects re-added)
            f' + {controls_str}'
            f' | {industry_col} + {time_col}'
        )

    extra_cols = [time_col, firm_col, industry_col, performance_metric, f'{performance_metric}_quintile']
    df = _prepare_df(df, outcome_col, map_var, CONTROLS, extra_cols)

    # Winsorize the outcome variable at 1% and 99% to mitigate outliers (after log-transform if applicable)
    df[outcome_col] = winsorize(df[outcome_col], limits=[0.01, 0.01], nan_policy='omit')

    # winsorize the control variables at 1% and 99% to mitigate outliers (after log-transform if applicable)
    for control in CONTROLS:
        df[control] = winsorize(df[control], limits=[0.01, 0.01], nan_policy='omit')
    
    # winsorize the performance metric at 1% and 99% to mitigate outliers (after log-transform if applicable)
    df[performance_metric] = winsorize(df[performance_metric], limits=[0.01, 0.01], nan_policy='omit')

    model = pf.feols(formula, data=df, vcov={'CRV1': firm_col})
    return model
# Helper function to extract results directly from pyfixest model attributes and format them as a string for display or saving. If the model does not have the expected pyfixest attributes, it falls back to using the summary text (e.g. for statsmodels models). 
def model_to_string(model, title=''):
    '''Extract results directly from pyfixest model attributes.'''
    import pandas as pd
    
    try:
        # pyfixest: build string from tidy results dataframe rounded to 3 decimals
        tidy = model.tidy().round(3)  # returns clean DataFrame with all stats
        meta = (
            f'Dep. Var:      {model._depvar}\n'
            f'Fixed Effects: {model._fixef}\n'
            f'Observations:  {model._N}\n'
            f'Number of firms: {model._data.Instrument.nunique()}\n'
            f'R2:            {model._r2:.4f}\n'
            f'Adj. R2:      {model._adj_r2:.4f}\n'
            f'Adj. R2 Within:     {model._adj_r2_within:.4f}\n'
            f'RMSE:          {model._rmse:.4f}\n'
        )
        return f'{title}\n{meta}\n{tidy.to_string()}\n'
    
    except AttributeError:
        # statsmodels fallback
        return f'{title}\n{model.summary().as_text()}\n'


Lastly, we can loop through the different performance metrics and perform the predictive validation regressions.

In [ ]:
# outcome: ROA
# main independent variable: MAP_fit_1, MAP_fit_2, MAP_fit_1_explicit, MAP_fit_2_explicit, MAP_fit_1_explicit_FT, MAP_fit_2_explicit_FT, MAP_fit_1_implicit, MAP_fit_2_implicit, MAP_fit_1_implicit_FT, MAP_fit_2_implicit_FT
CONTROLS = [
    'Size', 'Leverage', 'Growth', 'Age_log',
    'Return', 'Top1shareholder', 'Duality', 'Volatility'
]

CONTROLS_DIF = [
    'Size', 'Leverage', 'Growth', 'Return', 'Top1shareholder', 'Duality', 'Volatility'
]

# Set the number of quartiles for the quartile regressions
n_quartiles = 5

# It is also looked at the changes by running regressions with first differences instead of levels. This allows us to check whether the results are driven by unobserved time-invariant 
# firm characteristics (which are differenced out in the first difference regression) and to check whether the results are consistent when looking at changes instead of levels.

diff_cols = ['Operating_Margin', 'EPS', 'ROA'] + [col for col in analysis_df.columns if col.startswith('MAP_fit')] + CONTROLS_DIF

diff_df = (
    analysis_df[['Instrument', 'Period_End_Year', 'NAICS_Sector_Name'] + diff_cols]
    .sort_values(['Instrument', 'Period_End_Year'])
    .copy()
)

# Calculate the first difference of the specified columns within each firm (Instrument)
# Note that each instrument might not have all years of data, so just calculate the difference for a observation if both the current and previous year exist for that instrument

for id in diff_df['Instrument'].unique():
    # Get the subset of the dataframe for the current instrument
    instrument_df = diff_df[diff_df['Instrument'] == id].sort_values('Period_End_Year')
    # Loop through each year for the current instrument
    for year in instrument_df['Period_End_Year'].unique():
        # Get the current and previous year data for the instrument
        current_year_data = instrument_df[instrument_df['Period_End_Year'] == year]
        previous_year_data = instrument_df[instrument_df['Period_End_Year'] == year - 1]
        #check if the previous year exists for the same instrument
        if not previous_year_data.empty:
            # Calculate the first difference for the specified columns
            for col in diff_cols:
                diff_df.loc[
                    (diff_df['Instrument'] == id) & (diff_df['Period_End_Year'] == year), col
                ] = current_year_data[col].values[0] - previous_year_data[col].values[0]
        else:
            # If the previous year does not exist, set the first difference to NaN
            for col in diff_cols:
                diff_df.loc[
                    (diff_df['Instrument'] == id) & (diff_df['Period_End_Year'] == year), col
                ] = np.nan

diff_df = diff_df.dropna(subset=diff_cols).reset_index(drop=True)


if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    map_var_1 = 'MAP_fit_1'
    map_var_2 = 'MAP_fit_2'
    # Run regression analyses for each outcome variable and each MAP-fit measure and save the results in a dataframe
    for outcome_var in ['ROA', 'EPS', 'Operating_Margin']:
        # First, run the regressions with firm fixed effects and then with firm and industry fixed effects to check whether the results are consistent across different model specifications
        result_1_fe = run_validation_regression(analysis_df, outcome_var, map_var_1, firm_FE=True, industry_FE=False)
        result_2_fe = run_validation_regression(analysis_df, outcome_var, map_var_2, firm_FE=True, industry_FE=False)

        # Then, run the regressions with the lagged outcome variable to check whether the results are consistent when looking at changes instead of levels.
        result_lagged_1 = run_validation_regression_lagged(analysis_df, outcome_var, f'{outcome_var}_lag_t_minus_1', map_var_1)
        result_lagged_2 = run_validation_regression_lagged(analysis_df, outcome_var, f'{outcome_var}_lag_t_minus_1', map_var_2)

        # Alternatively, we could also run the regressions with first differences instead of levels to check whether the results are consistent when looking at changes instead of levels.
        result_diff_1 = run_validation_regression(diff_df, outcome_var, map_var_1, controls=CONTROLS_DIF, firm_FE=True, industry_FE=False)
        result_diff_2 = run_validation_regression(diff_df, outcome_var, map_var_2, controls=CONTROLS_DIF, firm_FE=True, industry_FE=False)

        # Next, we run the quartile regressions to check whether the results are consistent across different levels of the outcome variable and to check whether the results are driven by specific quantiles of the outcome variable (e.g. firms with very low or very high profitability).
        result_quartile_1, obs_per_quartile_1, mean_per_quartile_1, range_per_quartile_1 = run_validation_regression_quartiles(analysis_df, outcome_var, map_var_1, firm_FE=True, quartiles=n_quartiles)
        result_quartile_2, obs_per_quartile_2, mean_per_quartile_2, range_per_quartile_2 = run_validation_regression_quartiles(analysis_df, outcome_var, map_var_2, firm_FE=True, quartiles=n_quartiles)    

        # Save the coefficients, standard errors, t-values, and p-values of all variables to a dataframe
        coef_df_1_fe = result_1_fe.tidy().reset_index()
        coef_df_2_fe = result_2_fe.tidy().reset_index()
        coef_df_lagged_1 = result_lagged_1.tidy().reset_index()
        coef_df_lagged_2 = result_lagged_2.tidy().reset_index()
        coef_df_diff_1 = result_diff_1.tidy().reset_index()
        coef_df_diff_2 = result_diff_2.tidy().reset_index()
        coef_df_quartile_1 = result_quartile_1.tidy().reset_index()
        coef_df_quartile_2 = result_quartile_2.tidy().reset_index()

        # Define output path for the regression results
        if without_finance:
            output_path_1 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_MAP_fit_1_{measurement_method}_without_finance.txt'
            output_path_2 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_MAP_fit_2_{measurement_method}_without_finance.txt'
        elif within_industry:
            output_path_1 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_MAP_fit_1_{measurement_method}_within_industry.txt'
            output_path_2 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_MAP_fit_2_{measurement_method}_within_industry.txt'
        else:
            output_path_1 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_MAP_fit_1_{measurement_method}.txt'
            output_path_2 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_MAP_fit_2_{measurement_method}.txt'

        # save the summary statistics of the regression results of each map measure into a separat text file
        with open(output_path_1, 'w') as f:
            f.write(f'Results for MAP_fit_1 on {outcome_var} using Firm and Year Fixed Effects:\n\n')
            f.write(model_to_string(result_1_fe))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_1 on {outcome_var} using Lagged Outcome Variable:\n\n')
            f.write(model_to_string(result_lagged_1))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_1 on {outcome_var} using First Differences:\n\n')
            f.write(model_to_string(result_diff_1))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_1 on {outcome_var} using Quartile Regressions:\n\n')
            # shortly describe the quartile with number of observations in each quartile and the range of the outcome variable in each quartile
            # write the number of observations in each quartile
            for i in range(1, n_quartiles + 1):
                f.write(f'Quartile {i}: {obs_per_quartile_1.iloc[i-1]} observations, range of {map_var}: {range_per_quartile_1.iloc[i-1]}, mean: {mean_per_quartile_1.iloc[i-1]}\n')
            f.write('\n')
            # write the results of the quartile regression
            f.write(model_to_string(result_quartile_1))
            f.write('\n\n')

        with open(output_path_2, 'w') as f:
            f.write(f'Results for MAP_fit_2 on {outcome_var} using Firm and Year Fixed Effects:\n\n')
            f.write(model_to_string(result_2_fe))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_2 on {outcome_var} using Lagged Outcome Variable:\n\n')
            f.write(model_to_string(result_lagged_2))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_2 on {outcome_var} using First Differences:\n\n')
            f.write(model_to_string(result_diff_2))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_2 on {outcome_var} using Quartile Regressions:\n\n')
            # shortly describe the quartile with number of observations in each quartile and the range of the outcome variable in each quartile
            # write the number of observations in each quartile
            for i in range(1, n_quartiles + 1):
                f.write(f'Quartile {i}: {obs_per_quartile_2.iloc[i-1]} observations, range of {map_var}: {range_per_quartile_2.iloc[i-1]}, mean: {mean_per_quartile_2.iloc[i-1]}\n')
            f.write('\n')
            # write the results of the quartile regression
            f.write(model_to_string(result_quartile_2))
            f.write('\n\n')

    # Now we run the regression analyses for the lagged outcome variables for ROA, EPS, and Operating Margin
    for outcome_var in ['ROA_lag_t_plus_1', 'ROA_lag_t_plus_2', 'ROA_lag_t_plus_3', 'ROA_lag_t_plus_4', 'ROA_lag_t_plus_5', 'ROA_lag_t_plus_6',
                        'EPS_lag_t_plus_1', 'EPS_lag_t_plus_2', 'EPS_lag_t_plus_3', 'EPS_lag_t_plus_4', 'EPS_lag_t_plus_5', 'EPS_lag_t_plus_6',
                        'Operating_Margin_lag_t_plus_1', 'Operating_Margin_lag_t_plus_2', 'Operating_Margin_lag_t_plus_3', 'Operating_Margin_lag_t_plus_4',
                        'Operating_Margin_lag_t_plus_5', 'Operating_Margin_lag_t_plus_6']:

        result_1_fe_1 = run_validation_regression_persistence(analysis_df, outcome_var, map_var_1, firm_FE=True)
        result_1_fe_1_2 = run_validation_regression_persistence_quartiles(analysis_df, outcome_var, map_var_1, firm_FE=True, quartiles=5)
        result_1_fe_2 = run_validation_regression_persistence_heterogeneity(analysis_df, outcome_var, map_var_1, firm_FE=True)

        result_2_fe_1 = run_validation_regression_persistence(analysis_df, outcome_var, map_var_2, firm_FE=True)
        result_2_fe_1_2 = run_validation_regression_persistence_quartiles(analysis_df, outcome_var, map_var_2, firm_FE=True, quartiles=5)
        result_2_fe_2 = run_validation_regression_persistence_heterogeneity(analysis_df, outcome_var, map_var_2, firm_FE=True)

        # Save the coefficients, standard errors, t-values, and p-values of all variables to a dataframe

        coef_df_1_fe_1 = result_1_fe_1.tidy().reset_index()
        coef_df_1_fe_1_2 = result_1_fe_1_2.tidy().reset_index()
        coef_df_1_fe_2 = result_1_fe_2.tidy().reset_index()
        coef_df_2_fe_1 = result_2_fe_1.tidy().reset_index()
        coef_df_2_fe_1_2 = result_2_fe_1_2.tidy().reset_index()
        coef_df_2_fe_2 = result_2_fe_2.tidy().reset_index()

        # Define output path for the regression results
        if without_finance:
            output_path_1 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_MAP_fit_1_{measurement_method}_without_finance.txt'
            output_path_2 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_MAP_fit_2_{measurement_method}_without_finance.txt'
        elif within_industry:
            output_path_1 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_MAP_fit_1_{measurement_method}_within_industry.txt'
            output_path_2 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_MAP_fit_2_{measurement_method}_within_industry.txt'
        else:
            output_path_1 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_MAP_fit_1_{measurement_method}.txt'
            output_path_2 = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_MAP_fit_2_{measurement_method}.txt'

        # add the summary statistics of the regression results of each map measure to the text file above
        with open(output_path_1, 'a') as f:
            f.write(f'Results for MAP_fit_1 on {outcome_var} using Firm and Year Fixed Effects (Persistence 1):\n\n')
            f.write(model_to_string(result_1_fe_1))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_1 on {outcome_var} using Firm and Year Fixed Effects (Persistence 1 Quartiles):\n\n')
            f.write(model_to_string(result_1_fe_1_2))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_1 on {outcome_var} using Firm and Year Fixed Effects (Persistence 2):\n\n')
            f.write(model_to_string(result_1_fe_2))
        with open(output_path_2, 'a') as f:
            f.write(f'Results for MAP_fit_2 on {outcome_var} using Firm and Year Fixed Effects (Persistence 1):\n\n')
            f.write(model_to_string(result_2_fe_1))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_2 on {outcome_var} using Firm and Year Fixed Effects (Persistence 1 Quartiles):\n\n')
            f.write(model_to_string(result_2_fe_1_2))
            f.write('\n\n')
            f.write(f'Results for MAP_fit_2 on {outcome_var} using Firm and Year Fixed Effects (Persistence 2):\n\n')
            f.write(model_to_string(result_2_fe_2))

else:
    map_vars = ['MAP_fit_1_explicit', 'MAP_fit_2_explicit', 'MAP_fit_1_explicit_FT', 'MAP_fit_2_explicit_FT', 'MAP_fit_1_implicit', 'MAP_fit_2_implicit', 'MAP_fit_1_implicit_FT', 'MAP_fit_2_implicit_FT']
    
    for map_var in map_vars:
        for outcome_var in ['ROA', 'EPS', 'Operating_Margin']:

            result_fe = run_validation_regression(analysis_df, outcome_var, map_var, firm_FE=True, industry_FE=False)
            result_lagged = run_validation_regression_lagged(analysis_df, outcome_var, f'{outcome_var}_lag_t_minus_1', map_var)
            result_diff = run_validation_regression(diff_df, outcome_var, map_var, controls=CONTROLS_DIF, firm_FE=True, industry_FE=False)
            result_quartile, obs_per_quartile, mean_per_quartile, range_per_quartile = run_validation_regression_quartiles(analysis_df, outcome_var, map_var, firm_FE=True, quartiles=n_quartiles)

            # Save the coefficients, standard errors, t-values, and p-values of all variables to a dataframe
            coef_df_fe = result_fe.tidy().reset_index()
            coef_df_lagged = result_lagged.tidy().reset_index()
            coef_df_diff = result_diff.tidy().reset_index()
            coef_df_quartile = result_quartile.tidy().reset_index()

            # Define output path for the regression results
            if without_finance:
                output_path = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_{map_var}_{measurement_method}_without_finance.txt'
            elif within_industry:
                output_path = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_{map_var}_{measurement_method}_within_industry.txt'
            else:
                output_path = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var}_{map_var}_{measurement_method}.txt'

            # save the summary statistics of the regression results of each map measure into a separat text file
            with open(output_path, 'w') as f:
                f.write(f'Results for {map_var} on {outcome_var} using Firm and Year Fixed Effects:\n\n')
                f.write(model_to_string(result_fe))
                f.write('\n\n')
                f.write(f'Results for {map_var} on {outcome_var} using Lagged Outcome Variable:\n\n')
                f.write(model_to_string(result_lagged))
                f.write('\n\n')
                f.write(f'Results for {map_var} on {outcome_var} using First Differences:\n\n')
                f.write(model_to_string(result_diff))
                f.write('\n\n')
                f.write(f'Results for {map_var} on {outcome_var} using Quartile Regressions:\n\n')
                # shortly describe the quartile with number of observations in each quartile and the range of the outcome variable in each quartile
                # write the number of observations in each quartile
                f.write(f'Number of observations in each quartile:\n')
                for i in range(1, n_quartiles + 1):
                    f.write(f'Quartile {i}: {obs_per_quartile.iloc[i-1]} observations, range of {map_var}: {range_per_quartile.iloc[i-1]}, mean: {mean_per_quartile.iloc[i-1]}\n')
                f.write('\n')
                # write the results of the quartile regression
                f.write(model_to_string(result_quartile))
                f.write('\n\n')

        for outcome_var in ['ROA_lag_t_plus_1', 'ROA_lag_t_plus_2', 'ROA_lag_t_plus_3', 'ROA_lag_t_plus_4', 'ROA_lag_t_plus_5', 'ROA_lag_t_plus_6',
                        'EPS_lag_t_plus_1', 'EPS_lag_t_plus_2', 'EPS_lag_t_plus_3', 'EPS_lag_t_plus_4', 'EPS_lag_t_plus_5', 'EPS_lag_t_plus_6',
                        'Operating_Margin_lag_t_plus_1', 'Operating_Margin_lag_t_plus_2', 'Operating_Margin_lag_t_plus_3', 'Operating_Margin_lag_t_plus_4',
                        'Operating_Margin_lag_t_plus_5', 'Operating_Margin_lag_t_plus_6']:
            
            result_fe_1 = run_validation_regression_persistence(analysis_df, outcome_var, map_var, firm_FE=True)
            result_fe_1_2 = run_validation_regression_persistence_quartiles(analysis_df, outcome_var, map_var, firm_FE=True, quartiles=5)
            result_fe_2 = run_validation_regression_persistence_heterogeneity(analysis_df, outcome_var, map_var, firm_FE=True) 

            # Save the coefficients, standard errors, t-values, and p-values of all variables to a dataframe
            coef_df_fe_1 = result_fe_1.tidy().reset_index()
            coef_df_fe_1_2 = result_fe_1_2.tidy().reset_index()
            coef_df_fe_2 = result_fe_2.tidy().reset_index()

            # Define output path for the regression results
            if without_finance:
                output_path = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_{map_var}_{measurement_method}_without_finance.txt'
            elif within_industry:
                output_path = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_{map_var}_{measurement_method}_within_industry.txt'
            else:
                output_path = f'{tables_dir}/{measurement_approach}/Predictive_Regression_Results_{outcome_var.split("_lag_")[0]}_{map_var}_{measurement_method}.txt'

            # save the summary statistics of the regression results of each map measure into a separat text file
            with open(output_path, 'a') as f:
                f.write(f'Results for {map_var} on {outcome_var} using Industry and Year Fixed Effects (Persistence 1):\n\n')
                f.write(model_to_string(result_fe_1))
                f.write('\n\n')
                f.write(f'Results for {map_var} on {outcome_var} using Industry and Year Fixed Effects (Persistence 1 Quartiles):\n\n')
                f.write(model_to_string(result_fe_1_2))
                f.write('\n\n')
                f.write(f'Results for {map_var} on {outcome_var} using Industry and Year Fixed Effects (Persistence 2):\n\n')
                f.write(model_to_string(result_fe_2))


if measurement_approach == 'BoW' or measurement_approach.startswith('W2V'):
    del result_1_fe, result_2_fe, result_lagged_1, result_lagged_2, coef_df_1_fe, coef_df_2_fe, coef_df_lagged_1, coef_df_lagged_2, map_var_1, map_var_2, outcome_var, result_1_fe_1, result_1_fe_2, result_2_fe_1, result_2_fe_2, coef_df_1_fe_1, coef_df_1_fe_2, coef_df_2_fe_1, coef_df_2_fe_2, result_diff_1, result_diff_2, coef_df_diff_1, coef_df_diff_2, result_quartile_1, result_quartile_2, coef_df_quartile_1, coef_df_quartile_2, obs_per_quartile_1, obs_per_quartile_2, mean_per_quartile_1, mean_per_quartile_2, range_per_quartile_1, range_per_quartile_2, result_1_fe_1_2, result_2_fe_1_2, coef_df_1_fe_1_2, coef_df_2_fe_1_2
else:
    del result_fe, result_lagged, coef_df_fe, coef_df_lagged, map_var, result_fe_1, result_fe_2, coef_df_fe_1, coef_df_fe_2, result_diff, coef_df_diff, result_quartile, coef_df_quartile, obs_per_quartile, mean_per_quartile, range_per_quartile

del diff_df

# Additions - TODO

Add the Wald-test to the quartile regression analyses. This serves to check whether the coefficients on the different quartiles are significantly different.

In [ ]:
#print(pf.etable(model, type='tex' , signif_code = [0.01, 0.05, 0.10] , digits = 4, file_name = f'Tables_new/Validation/{measurement_type}/LaTeX/Summary_Statistics_{outcome_var.split('_lag_')[0]}_{map_var}_{measurement_method}.tex' ))

In [ ]:
# quartile regression
result_quartile_1, obs_per_quartile, mean_per_quartile, range_per_quartile = run_validation_regression_quartiles(analysis_df, 'Operating_Margin', 'MAP_fit_1_explicit', time_col='filing_year', outcome_log_transform=False, firm_FE=True, quartiles=5)
result_quartile_2 = run_validation_regression_persistence_quartiles(analysis_df, 'Operating_Margin_lag_t_plus_2', 'MAP_fit_1_explicit', outcome_log_transform=False, firm_FE=True, quartiles=5)
print('Results for MAP_fit_1 on Operating_Margin using Quartile Regressions:')
print(result_quartile_1.summary())
print('The Kurtosis of the residuals of the quartile regression is:', pd.Series(np.asarray(result_quartile_1.resid())).kurtosis())


In [ ]:
# test for similarity of coefficients across quartiles using wald test (e.g. MAP_fit_1_explicit_Q_2 = MAP_fit_1_explicit_Q_3)

#result_quartile_2.summary()

# Example restriction matrix for one linear constraint
R = np.array([[0, 1, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0]])  # adjust to match coefficient order

q = np.array([0])  # the value that the linear combination of coefficients should equal

result_quartile_2.wald_test(R, q)

In [ ]:
R = np.array([
    [1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],   # Q2 = Q3
    [0, 1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0],   # Q3 = Q4
    [0, 0, 1, -1, 0, 0, 0, 0, 0, 0, 0, 0],   # Q4 = Q5
])
q = np.array([0, 0, 0])

result_quartile_2.wald_test(R, q)